# 🏭 Тестирование модуля построения модели (3-Statement Model)

**Упрощенная и структурированная версия** с учетом доработок канонической структуры данных.

## 📋 Структура ноутбука:

1. **Инициализация и настройка** - загрузка конфигурации, инициализация DataMart
2. **Загрузка и валидация данных** - загрузка истории, валидация канонической структуры
3. **Построение модели** - запуск модели, проверка сходимости
4. **Валидация результатов** - проверка BS Identity, CF Identity, Retained Earnings
5. **Анализ и отчетность** - сводка результатов, выявление проблем

---

**Примечание**: Этот ноутбук использует модули `engine.acceptance.checks` и `engine.model.core` для валидации и построения модели, что исключает дублирование кода.

### 🔍 Детальный анализ сопоставимости истории и прогноза

**Примечание**: Этот раздел использует функции загрузки данных из DataMart. 
Для построения модели используйте `engine.model.core.build_model()`, 
для валидации - `engine.acceptance.checks.run_acceptance()`.

Анализ каждой статьи с проверкой:
- Знаков (положительные/отрицательные)
- Порядка чисел (масштаб)
- Логики изменений
- Сопоставимости величин

## 🔧 Импорты и настройка <a id="section-setup"></a>

In [1]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
import sys
from IPython.display import display, Markdown, HTML
import matplotlib.pyplot as plt
from typing import Optional

# Настройка для графиков в Jupyter
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')

# Определение корня проекта
current_dir = Path.cwd()
if (current_dir / 'engine').exists():
    ROOT = current_dir
elif (current_dir.parent / 'engine').exists():
    ROOT = current_dir.parent
elif (current_dir.parent.parent / 'engine').exists():
    ROOT = current_dir.parent.parent
else:
    nb_path = Path(__file__) if '__file__' in globals() else Path.cwd()
    ROOT = nb_path.parent.parent.parent

print(f"ROOT: {ROOT}")
print(f"Engine exists: {(ROOT / 'engine').exists()}")

# Добавляем корень проекта в sys.path ПЕРЕД импортом engine
sys.path.insert(0, str(ROOT))

# Определяем COMPANY автоматически из пути ноутбука
COMPANY = 'us_steel'  # fallback по умолчанию
if 'companies' in current_dir.parts:
    companies_idx = current_dir.parts.index('companies')
    if companies_idx + 1 < len(current_dir.parts):
        COMPANY = current_dir.parts[companies_idx + 1]

croot = ROOT / 'companies' / COMPANY
print(f"Company root: {croot}")

# Импорт engine модулей ПОСЛЕ настройки sys.path
from engine.database.data_mart import get_data_mart

MODEL_VERSION: Optional[str] = None

def is_history_year(year):
    """Проверяет входит ли год в исторический диапазон"""
    if 'HISTORY_START_YEAR' in globals() and 'HISTORY_END_YEAR' in globals():
        return HISTORY_START_YEAR <= year <= HISTORY_END_YEAR
    return 2010 <= year <= 2024

def get_history_years(data_dict):
    """Возвращает отсортированный список лет из словаря"""
    return sorted([y for y in data_dict.keys() if is_history_year(y)])

def _open_data_mart():
    try:
        return get_data_mart(ROOT, COMPANY)
    except Exception as exc:
        print(f"⚠️ Не удалось подключиться к витрине данных: {exc}")
        return None

def _resolve_model_version(mart):
    global MODEL_VERSION
    if MODEL_VERSION:
        return MODEL_VERSION
    try:
        versions = mart.get_existing_versions()
        if versions:
            MODEL_VERSION = versions[0]['version']
            return MODEL_VERSION
    except Exception as exc:
        print(f"⚠️ Не удалось определить версию модели: {exc}")
    return None

def load_history_from_db(statement_type='IS', canonical=False):
    stmt = statement_type.upper()
    df = pd.DataFrame()
    mart = _open_data_mart()
    if mart is not None:
        try:
            if stmt == 'IS':
                df = mart.get_history_income_statement(canonical=canonical)
            elif stmt == 'BS':
                df = mart.get_history_balance_sheet(canonical=canonical)
            else:
                df = mart.get_history_cash_flow(canonical=canonical)
            if df is not None and not df.empty:
                print(f"✅ История {stmt} загружена из витрины")
        except Exception as exc:
            print(f"⚠️ Не удалось загрузить историю {stmt} из витрины: {exc}")
        finally:
            mart.close()
    file_map = {
        'IS': f"history/is_history_{COMPANY}.csv",
        'BS': f"history/bs_history_{COMPANY}.csv",
        'CF': f"history/cf_history_{COMPANY}.csv"
    }
    if df is None or df.empty:
        csv_path = croot / file_map.get(stmt, '')
        if csv_path.exists():
            print(f"📄 История {stmt} загружена из CSV")
            return pd.read_csv(csv_path)
        print(f"❌ История {stmt} не найдена ни в витрине, ни в CSV")
        return pd.DataFrame()
    if not canonical and 'metric' in df.columns:
        year_cols = [c for c in df.columns if str(c).isdigit()]
        if year_cols:
            df_long = df.set_index('metric')[year_cols].T.reset_index().rename(columns={'index': 'year'})
            df_long['year'] = df_long['year'].astype(int)
            return df_long
    return df

def load_model_table(table_name: str, canonical: bool = False, csv_filename: Optional[str] = None):
    table_key = table_name.upper()
    df = pd.DataFrame()
    mart = _open_data_mart()
    if mart is not None:
        version = _resolve_model_version(mart)
        if version:
            try:
                df = mart.get_model_results(version, table_key, canonical=canonical)
                if df is not None and not df.empty:
                    print(f"✅ Таблица {table_key} загружена из витрины (версия {version})")
                    mart.close()
                    return df
            except Exception as exc:
                print(f"⚠️ Не удалось загрузить таблицу {table_key} из витрины: {exc}")
            finally:
                mart.close()
        else:
            mart.close()
    if csv_filename:
        csv_path = croot / csv_filename
        if csv_path.exists():
            print(f"⚠️ Используем legacy CSV для {table_key}: {csv_path.relative_to(ROOT)}")
            return pd.read_csv(csv_path)
    print(f"⚠️ Таблица {table_key} не найдена ни в витрине, ни в CSV")
    return pd.DataFrame()

def load_model_forecast_from_db(statement_type='IS', canonical=True):
    stmt = statement_type.upper()
    csv_filename = f"outputs/model/3statement_{stmt}.csv"
    return load_model_table(stmt, canonical=canonical, csv_filename=csv_filename)

def load_model_parameter_table(parameter_name: str, csv_filename: Optional[str] = None):
    return load_model_table(parameter_name.upper(), canonical=False, csv_filename=csv_filename)

def load_company_csv(relative_path: str):
    csv_path = croot / relative_path
    if csv_path.exists():
        print(f"⚠️ Используем legacy CSV: {csv_path.relative_to(ROOT)}")
        return pd.read_csv(csv_path)
    print(f"❌ Legacy CSV не найден: {relative_path}")
    return pd.DataFrame()


ROOT: /home
Engine exists: True
Company root: /home/companies/us_steel


## 1️⃣ Проверка конфигурации <a id="section-1"></a>

In [2]:
display(Markdown("### Конфигурация из project.yaml"))

proj_yaml_path = croot / "configs" / "project.yaml"
if not proj_yaml_path.exists():
    print(f"❌ Файл конфигурации не найден: {proj_yaml_path}")
else:
    proj_yaml = yaml.safe_load(proj_yaml_path.read_text(encoding='utf-8'))
    
    print(f"✅ Конфигурация загружена")
    print(f"\n📊 Основные настройки:")
    print(f"  - Модель: {proj_yaml.get('model', {}).get('mode', 'standard')}")
    
    if 'model' in proj_yaml and 'standard' in proj_yaml['model']:
        std_cfg = proj_yaml['model']['standard']
        print(f"\n📈 Revenue:")
        rev_cfg = std_cfg.get('revenue', {})
        print(f"  - Driver: {rev_cfg.get('driver', 'N/A')}")
        method_str = 'elastic_net' if rev_cfg.get('use_elastic_net', False) else rev_cfg.get('fallback_growth', 'N/A')
        print(f"  - Method: {method_str}")
        print(f"  - Target Transform: {rev_cfg.get('target_transform', 'N/A')}")
        print(f"  - Feature Transform: {rev_cfg.get('feature_transform', 'N/A')}")
        print(f"  - Chain-link Anchor: {rev_cfg.get('chainlink_anchor', 'N/A')}")
        
        print(f"\n💰 WC:")
        wc_cfg = std_cfg.get('wc', {})
        print(f"  - Метод: {wc_cfg.get('method', 'N/A')}")
        print(f"  - DSO: {wc_cfg.get('dso_days', 'N/A')}")
        print(f"  - DIO: {wc_cfg.get('dio_days', 'N/A')}")
        print(f"  - DPO: {wc_cfg.get('dpo_days', 'N/A')}")
        print(f"  - Transform Days: {wc_cfg.get('transform_days', 'N/A')}")
        
        print(f"\n💳 Debt:")
        debt_cfg = std_cfg.get('debt', {})
        print(f"  - Interest treatment: {debt_cfg.get('interest_treatment', 'N/A')}")
        refinancing_enable = debt_cfg.get('refinancing', {}).get('enable', False)
        print(f"  - Refinancing enabled: {refinancing_enable}")
        rc_cfg = debt_cfg.get('rc', {})
        if rc_cfg.get('enable', False):
            print(f"  - RC enabled: True")
            print(f"  - RC limit: {rc_cfg.get('limit', 'N/A')}")
            print(f"  - Min cash: {rc_cfg.get('min_cash', 'N/A')}")
        
        print(f"\n📊 COGS:")
        cogs_cfg = std_cfg.get('cogs', {})
        print(f"  - Use PPI uplift: {cogs_cfg.get('use_ppi_uplift', False)}")
        clamp_cfg = cogs_cfg.get('clamp', {})
        if clamp_cfg:
            print(f"  - Clamp: [{clamp_cfg.get('min', 'N/A')}, {clamp_cfg.get('max', 'N/A')}]")
        
        print(f"\n🔬 Train/Test:")
        # Проверяем оба возможных места в конфиге
        tt_cfg_model = proj_yaml.get('model', {}).get('train_test', {})
        tt_cfg_std = std_cfg.get('train_test', {})
        tt_cfg = tt_cfg_model or tt_cfg_std
        train_end = tt_cfg.get('train_end_year', None) if tt_cfg else None
        if train_end:
            print(f"  - Train end year: {train_end}")
        else:
            print(f"  - Train end year: N/A (не настроен)")
        
        # ГЛОБАЛЬНЫЕ ПЕРЕМЕННЫЕ ДЛЯ ПЕРИОДОВ (для использования во всем ноутбуке)
        periods_config = proj_yaml.get("model", {}).get("standard", {}).get("periods", {})
        HISTORY_START_YEAR = periods_config.get("history_start_year", 2010)
        HISTORY_END_YEAR = periods_config.get("history_end_year", 2024)
        FORECAST_START_YEAR = periods_config.get("forecast_start_year", 2025)
        FORECAST_END_YEAR = periods_config.get("forecast_end_year", 2029)
        
        print(f"\n📅 Параметры периодов (глобальные переменные):")
        print(f"  - История: {HISTORY_START_YEAR}-{HISTORY_END_YEAR}")
        print(f"  - Прогноз: {FORECAST_START_YEAR}-{FORECAST_END_YEAR}")


### Конфигурация из project.yaml

✅ Конфигурация загружена

📊 Основные настройки:
  - Модель: custom

📈 Revenue:
  - Driver: steel_price_hrc
  - Method: elastic_net
  - Target Transform: dln
  - Feature Transform: dln
  - Chain-link Anchor: history_last

💰 WC:
  - Метод: days
  - DSO: 45
  - DIO: 60
  - DPO: 50
  - Transform Days: dln

💳 Debt:
  - Interest treatment: separate_line
  - Refinancing enabled: True
  - RC enabled: True
  - RC limit: 2000.0
  - Min cash: 500000000.0

📊 COGS:
  - Use PPI uplift: False
  - Clamp: [0.55, 0.98]

🔬 Train/Test:
  - Train end year: 2022

📅 Параметры периодов (глобальные переменные):
  - История: 2010-2024
  - Прогноз: 2025-2029


## 2️⃣ Загрузка входных данных <a id="section-2"></a>

In [3]:
display(Markdown("### Проверка наличия входных данных"))

history_checks = []
history_keys = {'IS': 'history/is_history_{company}.csv',
                'BS': 'history/bs_history_{company}.csv',
                'CF': 'history/cf_history_{company}.csv'}

mart = _open_data_mart()
db_exists = False
if mart is not None:
    try:
        hist_is_db = mart.get_history_income_statement(canonical=False)
        db_exists = hist_is_db is not None and not hist_is_db.empty
    except Exception as exc:
        print(f"⚠️ Не удалось прочитать историю из витрины: {exc}")

print(f"\n🗄️ Витрина данных: {'✅' if db_exists else '⚠️ (данных нет)'}")
for stmt_type, template in history_keys.items():
    entry = {
        'type': stmt_type,
        'source': 'Data Mart' if db_exists else 'CSV',
        'path': 'Data Mart' if db_exists else template.format(company=COMPANY)
    }
    if db_exists:
        entry['exists'] = '✅'
    else:
        csv_df = load_company_csv(template.format(company=COMPANY))
        entry['exists'] = '✅' if not csv_df.empty else '❌'
        if not csv_df.empty:
            entry['rows'] = len(csv_df)
            entry['cols'] = len(csv_df.columns)
    history_checks.append(entry)

display(pd.DataFrame(history_checks))

macro_rows = []
factors = []
project_cfg = {}
project_path = croot / "configs" / "project.yaml"
if project_path.exists():
    try:
        project_cfg = yaml.safe_load(project_path.read_text(encoding="utf-8")) or {}
    except Exception as exc:
        print(f"⚠️ Не удалось прочитать project.yaml для macro-факторов: {exc}")
macro_cfg = project_cfg.get("macro_forecast", {})
factors = macro_cfg.get("factors", [])
if mart is not None:
    for factor in factors or []:
        try:
            forecast = mart.get_macro_forecast(factor) or {}
        except Exception as exc:
            print(f"⚠️ Не удалось прочитать прогноз для {factor}: {exc}")
            forecast = {}
        macro_rows.append({
            'factor': factor,
            'years_available': len(forecast),
            'start_year': min(forecast) if forecast else None,
            'end_year': max(forecast) if forecast else None
        })
    mart.close()
    if macro_rows:
        display(Markdown("📊 Макро-прогнозы (витрина данных)"))
        display(pd.DataFrame(macro_rows))
else:
    print("⚠️ Витрина данных недоступна")


### Проверка наличия входных данных


🗄️ Витрина данных: ✅


,type,source,path,exists
0,IS,Data Mart,Data Mart,✅
1,BS,Data Mart,Data Mart,✅
2,CF,Data Mart,Data Mart,✅


📊 Макро-прогнозы (витрина данных)

,factor,years_available,start_year,end_year
0,gdp_us,5,2025,2029
1,industrial_production_us,5,2025,2029
2,dxy,5,2025,2029
3,steel_price_hrc,5,2025,2029
4,iron_ore_price,5,2025,2029
5,cpi_us,5,2025,2029
6,ppi_us,5,2025,2029
7,gdp_world,5,2025,2029


## 2.1 Анализ возможностей построения модели <a id="section-2-1"></a>

In [4]:
display(Markdown("### 🔍 Анализ доступных данных и возможностей моделирования"))

mart = _open_data_mart()
if mart is None:
    print("❌ Не удалось подключиться к витрине данных — пропускаем аналитические проверки.")
else:
    detected_model_type = mart.detect_model_type()
    config_model_mode = proj_yaml.get('model', {}).get('mode', 'standard')

    print(f"📋 Тип модели в конфиге: {config_model_mode}")
    print(f"🔍 Определенный тип модели из истории: {detected_model_type}")
    if detected_model_type != config_model_mode:
        print(f"⚠️  ВНИМАНИЕ: Тип модели в конфиге ({config_model_mode}) не совпадает с определенным из истории ({detected_model_type})")
        print(f"   Рекомендуется установить model.mode: {detected_model_type} в project.yaml")
    mart.close()



### 🔍 Анализ доступных данных и возможностей моделирования

📋 Тип модели в конфиге: custom
🔍 Определенный тип модели из истории: custom


### 2.1.1 Проверка доступности канонических метрик

In [5]:
mart = _open_data_mart()
if mart is None:
    print("❌ Витрина данных недоступна — пропускаем анализ доступности метрик.")
    availability_check = {'IS': {}, 'BS': {}, 'CF': {}}
else:
    standard_is = mart.get_canonical_metrics_from_db('IS')
    standard_bs = mart.get_canonical_metrics_from_db('BS')
    standard_cf = mart.get_canonical_metrics_from_db('CF')

    hist_is_df = mart.get_history_income_statement(canonical=True)
    hist_bs_df = mart.get_history_balance_sheet(canonical=True)
    hist_cf_df = mart.get_history_cash_flow(canonical=True)
    mart.close()

    def _wide_to_year_index(df):
        if df is None or df.empty or 'metric' not in df.columns:
            return pd.DataFrame()
        wide = df.set_index('metric')
        year_cols = [c for c in wide.columns if str(c).isdigit()]
        return wide[year_cols].T.apply(pd.to_numeric, errors='coerce')

    hist_is_df = _wide_to_year_index(hist_is_df)
    hist_bs_df = _wide_to_year_index(hist_bs_df)
    hist_cf_df = _wide_to_year_index(hist_cf_df)

    availability_check = {'IS': {}, 'BS': {}, 'CF': {}}

    def check_metric_availability(df, metric_name):
        if df is None or df.empty:
            return False, 0, []
        metric_lower = metric_name.lower()
        columns = {str(c).lower(): c for c in df.columns}
        if metric_lower not in columns:
            return False, 0, []
        col = columns[metric_lower]
        series = df[col].dropna()
        years = [int(y) for y, val in series.items() if pd.notna(val) and val != 0]
        return len(years) > 0, len(years), sorted(years)

    for metric in standard_is:
        has_data, count, years = check_metric_availability(hist_is_df, metric)
        availability_check['IS'][metric] = {'available': has_data, 'years_count': count, 'years': years}

    for metric in standard_bs:
        has_data, count, years = check_metric_availability(hist_bs_df, metric)
        availability_check['BS'][metric] = {'available': has_data, 'years_count': count, 'years': years}

    for metric in standard_cf:
        has_data, count, years = check_metric_availability(hist_cf_df, metric)
        availability_check['CF'][metric] = {'available': has_data, 'years_count': count, 'years': years}

    print(f"✅ Проверено метрик: IS={len(availability_check['IS'])}, BS={len(availability_check['BS'])}, CF={len(availability_check['CF'])}")



✅ Проверено метрик: IS=28, BS=51, CF=52


### 2.1.2 Чеклист возможностей построения модели

In [6]:
# Строим чеклист возможностей модели
display(Markdown("### 📋 Чеклист возможностей построения модели"))

checklist_data = []

# Проверяем возможность построения Standard модели
standard_required = {
    'IS': ['revenue', 'cogs', 'sga', 'ebitda', 'ebit', 'interest_expense', 'ebt', 'tax_expense', 'net_income'],
    'BS': ['cash', 'accounts_receivable', 'inventory', 'ppe_net', 'accounts_payable', 'short_term_debt', 'long_term_debt'],
    'CF': ['net_income', 'depreciation', 'wc_delta', 'cfo', 'capex', 'cfi', 'cff']
}

def can_build_model(model_type, required_metrics):
    """Проверяет возможность построения модели определенного типа"""
    missing = []
    partial = []
    
    for stmt_type, metrics in required_metrics.items():
        for metric in metrics:
            check = availability_check[stmt_type].get(metric, {})
            if not check.get('available', False):
                missing.append(f"{stmt_type}.{metric}")
            elif check.get('years_count', 0) < 3:
                partial.append(f"{stmt_type}.{metric} ({check['years_count']} лет)")
    
    can_build = len(missing) == 0
    quality = "high" if len(partial) == 0 else "medium" if len(partial) < len(required_metrics) * 0.3 else "low"
    
    return can_build, missing, partial, quality

# Проверяем Standard модель
can_standard, missing_standard, partial_standard, quality_standard = can_build_model('standard', standard_required)

# Проверяем Custom модель (без DTA/DTL)
custom_without_tax_required = standard_required.copy()
can_custom_wo_tax, missing_custom_wo_tax, partial_custom_wo_tax, quality_custom_wo_tax = can_build_model('custom_wo_tax', custom_without_tax_required)

# Проверяем Custom модель (с DTA/DTL)
custom_with_tax_required = standard_required.copy()
custom_with_tax_required['BS'].extend(['dta', 'dtl', 'taxes_payable'])
can_custom_with_tax, missing_custom_with_tax, partial_custom_with_tax, quality_custom_with_tax = can_build_model('custom_with_tax', custom_with_tax_required)

# Формируем таблицу результатов
checklist_rows = [
    {
        'Тип модели': 'Standard',
        'Можно строить': '✅ Да' if can_standard else '❌ Нет',
        'Качество данных': quality_standard,
        'Отсутствует': ', '.join(missing_standard[:5]) + ('...' if len(missing_standard) > 5 else ''),
        'Неполные данные': f"{len(partial_standard)} метрик"
    },
    {
        'Тип модели': 'Custom (без DTA/DTL)',
        'Можно строить': '✅ Да' if can_custom_wo_tax else '❌ Нет',
        'Качество данных': quality_custom_wo_tax,
        'Отсутствует': ', '.join(missing_custom_wo_tax[:5]) + ('...' if len(missing_custom_wo_tax) > 5 else ''),
        'Неполные данные': f"{len(partial_custom_wo_tax)} метрик"
    },
    {
        'Тип модели': 'Custom (с DTA/DTL)',
        'Можно строить': '✅ Да' if can_custom_with_tax else '❌ Нет',
        'Качество данных': quality_custom_with_tax,
        'Отсутствует': ', '.join(missing_custom_with_tax[:5]) + ('...' if len(missing_custom_with_tax) > 5 else ''),
        'Неполные данные': f"{len(partial_custom_with_tax)} метрик"
    }
]

checklist_df = pd.DataFrame(checklist_rows)
display(HTML(checklist_df.to_html(index=False, escape=False, classes='table table-striped')))

### 📋 Чеклист возможностей построения модели

Тип модели,Можно строить,Качество данных,Отсутствует,Неполные данные
Standard,✅ Да,high,,0 метрик
Custom (без DTA/DTL),✅ Да,high,,0 метрик
Custom (с DTA/DTL),✅ Да,high,,0 метрик


### 2.1.3 Методы моделирования статей и полнота данных


In [7]:
# Детальный анализ методов моделирования
display(Markdown("### 📊 Методы моделирования статей и полнота данных"))

modeling_methods = {
    'IS': {
        'revenue': 'Регрессия с макрофакторами / Темп роста / EWA',
        'cogs': 'PPI индексация / Ratio к Revenue / Clamp',
        'sga': 'EWA темпа роста / CPI индексация / Ratio к Revenue',
        'depreciation_owned': 'PP&E corkscrew (амортизация)',
        'depreciation_rou': 'Lease schedule (ROU амортизация)',
        'amortization': 'Intangibles corkscrew',
        'interest_expense': 'Debt schedule (расчет по ставкам)',
        'lease_interest': 'Lease schedule (проценты)',
        'tax_expense': 'Tax schedule (statutory rate / effective)',
    },
    'BS': {
        'cash': 'Cash bridge (CFF → CFI → CFO → Cash)',
        'accounts_receivable': 'WC days (DSO)',
        'inventory': 'WC days (DIO)',
        'accounts_payable': 'WC days (DPO)',
        'ppe_gross': 'PP&E corkscrew (Gross = Opening + CapEx - Disposals)',
        'ppe_accum_dep': 'PP&E corkscrew (Accumulated Depreciation)',
        'ppe_net': 'PP&E corkscrew (Net = Gross - Accum Dep)',
        'rou_asset': 'Lease schedule (ROU Asset)',
        'intangibles': 'Intangibles corkscrew',
        'short_term_debt': 'Debt schedule (ST часть)',
        'long_term_debt': 'Debt schedule (LT часть)',
        'current_lease_liability': 'Lease schedule (ST часть)',
        'long_term_lease_liability': 'Lease schedule (LT часть)',
        'dta': 'Tax schedule (Deferred Tax Assets)',
        'dtl': 'Tax schedule (Deferred Tax Liabilities)',
        'taxes_payable': 'Tax schedule (Taxes Payable)',
        'retained_earnings': 'Retained Earnings = Opening + Net Income - Dividends',
    },
    'CF': {
        'net_income': 'Из IS',
        'depreciation': 'Из PP&E corkscrew',
        'amortization': 'Из Intangibles corkscrew',
        'wc_delta': 'WC corkscrew (ΔAR + ΔInv - ΔAP - ΔOther)',
        'tax_paid': 'Tax schedule (Taxes Paid)',
        'cfo': 'Net Income + D&A - WC Delta - Tax Paid',
        'capex': 'CapEx policy (ratio / fixed / driver-based)',
        'disposal_proceeds': 'Disposals из PP&E corkscrew',
        'cfi': 'CapEx + Disposals + Other CFI',
        'debt_borrowings': 'Debt schedule (новые заимствования)',
        'debt_repayments': 'Debt schedule (погашения)',
        'lease_payments_cff': 'Lease schedule (основной долг)',
        'cff': 'Borrowings - Repayments - Lease Payments - Dividends',
        'net_change': 'CFO + CFI + CFF',
    }
}

# Формируем детальную таблицу
detailed_rows = []
for stmt_type in ['IS', 'BS', 'CF']:
    for metric, method in modeling_methods.get(stmt_type, {}).items():
        check = availability_check[stmt_type].get(metric, {})
        has_data = check.get('available', False)
        years_count = check.get('years_count', 0)
        
        # Определяем качество моделирования
        if has_data and years_count >= 5:
            quality = '🟢 Высокая (5+ лет истории)'
            can_model_precise = True
        elif has_data and years_count >= 3:
            quality = '🟡 Средняя (3-4 года истории)'
            can_model_precise = True
        elif has_data:
            quality = '🟠 Низкая (1-2 года истории)'
            can_model_precise = True
        else:
            quality = '🔴 Нет данных (моделирование через assumptions)'
            can_model_precise = False
        
        detailed_rows.append({
            'Отчет': stmt_type,
            'Метрика': metric,
            'Есть данные': '✅' if has_data else '❌',
            'Лет истории': years_count,
            'Качество': quality,
            'Метод моделирования': method
        })

detailed_df = pd.DataFrame(detailed_rows)
display(HTML(detailed_df.to_html(index=False, escape=False, classes='table table-striped table-sm', max_rows=50)))

### 📊 Методы моделирования статей и полнота данных

Отчет,Метрика,Есть данные,Лет истории,Качество,Метод моделирования
IS,revenue,✅,15,🟢 Высокая (5+ лет истории),Регрессия с макрофакторами / Темп роста / EWA
IS,cogs,✅,15,🟢 Высокая (5+ лет истории),PPI индексация / Ratio к Revenue / Clamp
IS,sga,✅,15,🟢 Высокая (5+ лет истории),EWA темпа роста / CPI индексация / Ratio к Revenue
IS,depreciation_owned,✅,15,🟢 Высокая (5+ лет истории),PP&E corkscrew (амортизация)
IS,depreciation_rou,✅,6,🟢 Высокая (5+ лет истории),Lease schedule (ROU амортизация)
IS,amortization,✅,15,🟢 Высокая (5+ лет истории),Intangibles corkscrew
IS,interest_expense,✅,15,🟢 Высокая (5+ лет истории),Debt schedule (расчет по ставкам)
IS,lease_interest,✅,6,🟢 Высокая (5+ лет истории),Lease schedule (проценты)
IS,tax_expense,✅,15,🟢 Высокая (5+ лет истории),Tax schedule (statutory rate / effective)
BS,cash,✅,15,🟢 Высокая (5+ лет истории),Cash bridge (CFF → CFI → CFO → Cash)


### 2.1.4 Рекомендации


In [8]:
# Сводка и рекомендации

recommendations = []

# Рекомендация по типу модели
if detected_model_type == 'custom':
    if can_custom_with_tax:
        recommendations.append("✅ Рекомендуется использовать **Custom модель с Tax Schedule** (DTA/DTL/Taxes Payable) для максимальной точности")
    elif can_custom_wo_tax:
        recommendations.append("✅ Рекомендуется использовать **Custom модель без Tax Schedule** (расширенная форма без DTA/DTL)")
    else:
        recommendations.append("⚠️ Custom модель не может быть построена. Рекомендуется использовать **Standard модель**")
elif detected_model_type == 'standard':
    if can_standard:
        recommendations.append("✅ Достаточно данных для **Standard модели**")
    else:
        recommendations.append("❌ Недостаточно данных для Standard модели. Проверьте наличие обязательных метрик.")

# Рекомендации по улучшению качества
total_metrics = len(availability_check['IS']) + len(availability_check['BS']) + len(availability_check['CF'])
available_metrics = sum(
    sum(1 for v in stmt.values() if v.get('available', False))
    for stmt in availability_check.values()
)
coverage = (available_metrics / total_metrics * 100) if total_metrics > 0 else 0

if coverage < 70:
    recommendations.append(f"⚠️ Покрытие данных: {coverage:.1f}%. Рекомендуется добавить больше исторических данных для повышения точности модели.")
elif coverage < 85:
    recommendations.append(f"✅ Покрытие данных: {coverage:.1f}%. Хорошее покрытие, можно улучшить, добавив недостающие метрики.")
else:
    recommendations.append(f"✅ Покрытие данных: {coverage:.1f}%. Отличное покрытие данных!")

missing_by_stmt = {
    stmt: sorted(metric for metric, info in metrics.items() if not info.get('available', False))
    for stmt, metrics in availability_check.items()
}

if any(missing_by_stmt.values()):
    details = []
    for stmt, metrics in missing_by_stmt.items():
        if not metrics:
            continue
        formatted = ', '.join(metrics[:5]) + ('…' if len(metrics) > 5 else '')
        details.append(f"- **{stmt}**: {formatted}")
    if details:
        recommendations.append(
            "📈 Чтобы повысить покрытие и уменьшить долю assumptions, добавьте исторические ряды:<br>"
            + "<br>".join(details)
        )
    if missing_by_stmt.get('CF'):
        recommendations.append(
            "💧 История по `CF.tax_paid`, `CF.debt_borrowings` и `CF.debt_repayments` позволит точнее моделировать налоговые платежи и долговые Cash Flow."
        )

# Проверка критических метрик
critical_missing = []
for metric in ['revenue', 'cogs', 'sga', 'cash', 'accounts_receivable', 'inventory', 'accounts_payable']:
    stmt = 'IS' if metric in ['revenue', 'cogs', 'sga'] else 'BS'
    check = availability_check[stmt].get(metric, {})
    if not check.get('available', False):
        critical_missing.append(f"{stmt}.{metric}")

if critical_missing:
    recommendations.append(f"❌ КРИТИЧНО: Отсутствуют обязательные метрики: {', '.join(critical_missing)}")

for rec in recommendations:
    display(Markdown(rec))

print()
print("📊 Итоговая статистика:")
print(f"  - Всего метрик в канонической форме: {total_metrics}")
print(f"  - Доступно метрик в истории: {available_metrics}")
print(f"  - Покрытие данных: {coverage:.1f}%")
print(f"  - Определенный тип модели: {detected_model_type}")
print(f"  - Тип модели в конфиге: {config_model_mode}")


✅ Рекомендуется использовать **Custom модель с Tax Schedule** (DTA/DTL/Taxes Payable) для максимальной точности

✅ Покрытие данных: 90.8%. Отличное покрытие данных!

📈 Чтобы повысить покрытие и уменьшить долю assumptions, добавьте исторические ряды:<br>- **IS**: other_expenses, rnd<br>- **BS**: short_term_investments, total_non_current_assets, total_non_current_liabilities<br>- **CF**: cfo_other, gain_on_equity_investee_transactions_cf, interest_paid, lease_interest_cfo, proceeds_from_cost_reimbursement_government_grants…

💧 История по `CF.tax_paid`, `CF.debt_borrowings` и `CF.debt_repayments` позволит точнее моделировать налоговые платежи и долговые Cash Flow.


📊 Итоговая статистика:
  - Всего метрик в канонической форме: 131
  - Доступно метрик в истории: 119
  - Покрытие данных: 90.8%
  - Определенный тип модели: custom
  - Тип модели в конфиге: custom


### 📊 Сравнительная таблица: Standard vs Custom модель

Эта таблица показывает, какие метрики входят в **Standard** модель, а какие добавляются в **Custom** модель.

> 📖 **Детальное сравнение:** См. `docs/MODEL_ENGINES_COMPARISON.md` для полной сравнительной таблицы по всем статьям и методам моделирования.


### 2.1.5 Метрики, доступные в Data Mart


In [9]:
# Получаем канонические метрики для Standard и Custom из БД
from engine.database.sqlite_wrapper import get_company_db

db = get_company_db(ROOT, COMPANY)
cursor = db.conn.cursor()

# Получаем метрики для Standard модели
cursor.execute("""
    SELECT statement_type, metric_name, metric_order
    FROM canonical_metrics
    WHERE model_type = 'standard'
    ORDER BY statement_type, metric_order
""")
standard_metrics = {}
for row in cursor.fetchall():
    stmt = row[0]
    metric = row[1]
    if stmt not in standard_metrics:
        standard_metrics[stmt] = []
    standard_metrics[stmt].append(metric)

# Получаем метрики для Custom модели
cursor.execute("""
    SELECT statement_type, metric_name, metric_order
    FROM canonical_metrics
    WHERE model_type = 'custom'
    ORDER BY statement_type, metric_order
""")
custom_metrics = {}
for row in cursor.fetchall():
    stmt = row[0]
    metric = row[1]
    if stmt not in custom_metrics:
        custom_metrics[stmt] = []
    custom_metrics[stmt].append(metric)

print(f"✅ Загружено метрик из БД:")
print(f"  Standard: IS={len(standard_metrics.get('IS', []))}, BS={len(standard_metrics.get('BS', []))}, CF={len(standard_metrics.get('CF', []))}")
print(f"  Custom: IS={len(custom_metrics.get('IS', []))}, BS={len(custom_metrics.get('BS', []))}, CF={len(custom_metrics.get('CF', []))}")

✅ Загружено метрик из БД:
  Standard: IS=28, BS=51, CF=52
  Custom: IS=28, BS=51, CF=52


### 2.1.6 Сравнительные таблицы источников данных

Сводим Standard и Custom наборы метрик по каждому отчётному блоку, чтобы убедиться, что Data Mart содержит полный объём данных для расширенной модели.


In [10]:
# Формируем сравнительную таблицу для каждого отчета
comparison_tables = []

for statement_type in ['IS', 'BS', 'CF']:
    standard_list = set(standard_metrics.get(statement_type, []))
    custom_list = set(custom_metrics.get(statement_type, []))
    
    # Определяем категории метрик
    in_both = sorted(standard_list & custom_list)
    only_custom = sorted(custom_list - standard_list)
    only_standard = sorted(standard_list - custom_list)
    
    # Формируем таблицу
    rows = []
    
    # Сначала метрики, которые в обеих моделях
    for metric in in_both:
        rows.append({
            'Отчет': statement_type,
            'Метрика': metric,
            'Standard': '✅',
            'Custom': '✅',
            'Категория': 'Обе модели'
        })
    
    # Затем метрики только в Standard (если есть)
    for metric in only_standard:
        rows.append({
            'Отчет': statement_type,
            'Метрика': metric,
            'Standard': '✅',
            'Custom': '❌',
            'Категория': 'Только Standard'
        })
    
    # Затем метрики только в Custom
    for metric in only_custom:
        rows.append({
            'Отчет': statement_type,
            'Метрика': metric,
            'Standard': '❌',
            'Custom': '✅',
            'Категория': 'Только Custom'
        })
    
    comparison_tables.append({
        'statement': statement_type,
        'rows': rows,
        'stats': {
            'in_both': len(in_both),
            'only_standard': len(only_standard),
            'only_custom': len(only_custom),
            'total_standard': len(standard_list),
            'total_custom': len(custom_list)
        }
    })

print(f"✅ Сформировано {len(comparison_tables)} сравнительных таблиц")


✅ Сформировано 3 сравнительных таблиц


#### 2.1.7 Сравнение состава метрик по отчётам

Ниже — развёрнутые таблицы для IS, BS и CF, показывающие, какие показатели присутствуют в стандартной и кастомной конфигурации модели.


In [11]:
# Выводим сравнительные таблицы для каждого отчета
for comp in comparison_tables:
    stmt = comp['statement']
    stats = comp['stats']
    
    display(Markdown(f"#### 📋 {stmt} - Income Statement" if stmt == 'IS' else f"#### 📋 {stmt} - Balance Sheet" if stmt == 'BS' else f"#### 📋 {stmt} - Cash Flow Statement"))
    
    # Статистика
    display(Markdown(f"""
    **Статистика:**
    - ✅ В обеих моделях: {stats['in_both']} метрик
    - 📊 Только в Standard: {stats['only_standard']} метрик
    - 🎯 Только в Custom: {stats['only_custom']} метрик
    - 📈 Всего Standard: {stats['total_standard']} метрик
    - 📈 Всего Custom: {stats['total_custom']} метрик
    """))
    
    # Таблица
    df_comp = pd.DataFrame(comp['rows'])
    display(HTML(df_comp.to_html(index=False, escape=False, classes='table table-striped table-bordered')))

#### 📋 IS - Income Statement


    **Статистика:**
    - ✅ В обеих моделях: 28 метрик
    - 📊 Только в Standard: 0 метрик
    - 🎯 Только в Custom: 0 метрик
    - 📈 Всего Standard: 28 метрик
    - 📈 Всего Custom: 28 метрик
    

Отчет,Метрика,Standard,Custom,Категория
IS,amortization,✅,✅,Обе модели
IS,asset_impairment_charges,✅,✅,Обе модели
IS,cogs,✅,✅,Обе модели
IS,depreciation_owned,✅,✅,Обе модели
IS,depreciation_rou,✅,✅,Обе модели
IS,ebit,✅,✅,Обе модели
IS,ebitda,✅,✅,Обе модели
IS,ebt,✅,✅,Обе модели
IS,eps_basic,✅,✅,Обе модели
IS,eps_diluted,✅,✅,Обе модели


#### 📋 BS - Balance Sheet


    **Статистика:**
    - ✅ В обеих моделях: 51 метрик
    - 📊 Только в Standard: 0 метрик
    - 🎯 Только в Custom: 0 метрик
    - 📈 Всего Standard: 51 метрик
    - 📈 Всего Custom: 51 метрик
    

Отчет,Метрика,Standard,Custom,Категория
BS,accounts_payable,✅,✅,Обе модели
BS,accounts_payable_related_parties,✅,✅,Обе модели
BS,accounts_receivable,✅,✅,Обе модели
BS,accrued_interest,✅,✅,Обе модели
BS,accrued_liabilities,✅,✅,Обе модели
BS,accrued_taxes,✅,✅,Обе модели
BS,aoci,✅,✅,Обе модели
BS,ap_related_parties,✅,✅,Обе модели
BS,apic,✅,✅,Обе модели
BS,cash,✅,✅,Обе модели


#### 📋 CF - Cash Flow Statement


    **Статистика:**
    - ✅ В обеих моделях: 52 метрик
    - 📊 Только в Standard: 0 метрик
    - 🎯 Только в Custom: 0 метрик
    - 📈 Всего Standard: 52 метрик
    - 📈 Всего Custom: 52 метрик
    

Отчет,Метрика,Standard,Custom,Категория
CF,active_employee_benefit_investments,✅,✅,Обе модели
CF,amortization,✅,✅,Обе модели
CF,asset_impairment_charges_cf,✅,✅,Обе модели
CF,capex,✅,✅,Обе модели
CF,cash_ending,✅,✅,Обе модели
CF,cash_opening,✅,✅,Обе модели
CF,cf_fx_effect,✅,✅,Обе модели
CF,cff,✅,✅,Обе модели
CF,cff_amortization_financing_costs,✅,✅,Обе модели
CF,cff_borrowings,✅,✅,Обе модели


#### 2.1.8 Сводка метрик только для Custom


In [12]:
# Сводная таблица: что добавляется в Custom
display(Markdown("### 🎯 Итого: Что добавляется в Custom модель"))

summary_rows = []
for comp in comparison_tables:
    stmt = comp['statement']
    only_custom_metrics = [
        r['Метрика'] for r in comp['rows'] 
        if r['Категория'] == 'Только Custom'
    ]
    
    if only_custom_metrics:
        for metric in only_custom_metrics:
            summary_rows.append({
                'Отчет': stmt,
                'Метрика (только в Custom)': metric
            })
    else:
        summary_rows.append({
            'Отчет': stmt,
            'Метрика (только в Custom)': '— Нет дополнительных метрик'
        })

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    display(HTML(summary_df.to_html(index=False, escape=False, classes='table table-striped table-bordered')))

### 🎯 Итого: Что добавляется в Custom модель

Отчет,Метрика (только в Custom)
IS,— Нет дополнительных метрик
BS,— Нет дополнительных метрик
CF,— Нет дополнительных метрик


## 2.2 Препроцессинг драйверов <a id="section-2-2"></a>

In [13]:
SUMMARY_YEAR = -1

with get_data_mart(ROOT, COMPANY) as mart:
    metrics_wc = mart.get_model_metrics("wc_days")
    metrics_margin = mart.get_model_metrics("margin_ratios")
    metrics_capex = mart.get_model_metrics("capex")
    tax_schedule_raw = mart.get_tax_schedule()


def _prepare_tax_schedule(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=["metric_name", "year", "value", "source", "meta"])
    prepared = df.copy()
    prepared["metric_name"] = prepared["metric"].astype(str)
    prepared["source"] = "tax_schedule"
    prepared["meta"] = [{} for _ in range(len(prepared))]
    return prepared[["metric_name", "year", "value", "source", "meta"]]


def display_metric_group(title: str, df: pd.DataFrame) -> None:
    display(Markdown(f"### {title}"))
    if df is None or df.empty:
        display(Markdown("⚠️ Метрики отсутствуют"))
        return
    df = df.copy()
    if "meta" not in df.columns:
        df["meta"] = [{} for _ in range(len(df))]
    summary = df[df.get("year") == SUMMARY_YEAR].copy()
    history = df[df.get("year") != SUMMARY_YEAR].copy() if "year" in df else pd.DataFrame()
    if not summary.empty:
        summary_display = summary[["metric_name", "value", "source", "meta"]].reset_index(drop=True)
        summary_display["meta"] = summary_display["meta"].apply(lambda x: x if isinstance(x, dict) else {})
        display(Markdown("**Рекомендации**"))
        display(summary_display)
    if not history.empty:
        pivot = history.pivot_table(values="value", index="year", columns="metric_name")
        display(Markdown("**Исторические значения**"))
        display(pivot.sort_index())


tax_schedule = _prepare_tax_schedule(tax_schedule_raw)

metric_groups = [
    ("2.2.1 Рабочий капитал (дни)", metrics_wc),
    ("2.2.2 Маржинальность и налоги", metrics_margin),
    ("2.2.3 CapEx и амортизация", metrics_capex),
    ("2.2.4 Tax Schedule", tax_schedule),
]

for title, data in metric_groups:
    display_metric_group(title, data)


### 2.2.1 Рабочий капитал (дни)

**Рекомендации**

,metric_name,value,source,meta
0,ccc_ewa,-2.684320,ewa,"{'halflife': 5, 'observations': 15}"
1,ccc_last,17.604515,last,{'year': 2024}
2,ccc_mean,-10.542789,mean,{'observations': 15}
3,ccc_recommended,-2.684320,ewa,"{'mean': -10.542788837907562, 'last': 17.60451..."
4,dih_ewa,45.546092,ewa,"{'halflife': 5, 'observations': 15}"
5,dih_last,56.281650,last,{'year': 2024}
6,dih_mean,38.307184,mean,{'observations': 15}
7,dih_recommended,45.546092,ewa,"{'mean': 38.30718428046885, 'last': 56.2816500..."
8,dpo_ewa,53.466059,ewa,"{'halflife': 5, 'observations': 15}"
9,dpo_last,67.522404,last,{'year': 2024}


**Исторические значения**

metric_name,ccc,dih,dpo,dso
year,,,,
2010,-16.872008,21.304201,39.016545,0.840336
2011,-15.179470,23.462294,39.376023,0.734259
2012,-15.853486,19.564663,36.106636,0.688488
2013,-13.694382,23.313874,38.309503,1.301248
2014,-18.437188,24.868651,44.187318,0.881479
2015,-17.515751,27.552733,46.259761,1.191277
2016,-33.734998,26.019952,60.763795,1.008845
2017,-44.827498,26.743373,72.166789,0.595918
2018,-10.120053,62.054449,72.792361,0.617859


...

**Исторические значения**

metric_name,capex_pct_revenue
year,
2010,0.038909
2011,0.042647
2012,0.040111
2013,0.029320
2014,0.029723
2015,0.049451
2016,0.033831
2017,0.041224
2018,0.070602


### 2.2.4 Tax Schedule

**Исторические значения**

metric_name,effectiveincometaxratecontinuingoperations
year,
2010,0.250000
2011,0.000000
2012,0.000000
2013,0.000000
2014,0.470588
2015,0.000000
2016,0.000000
2017,0.000000
2018,0.000000


## 3️⃣ Построение модели <a id="section-3"></a>

In [ ]:

from engine.model.core import build_model

print(f"Запуск построения модели для {COMPANY}...")
print(f"ROOT: {ROOT}")
print(f"Company: {COMPANY}")

proj_yaml = yaml.safe_load((croot / "configs" / "project.yaml").read_text(encoding='utf-8'))
periods_config = proj_yaml.get("model", {}).get("standard", {}).get("periods", {})

print(f"\n📋 Входные параметры модели:")
print(f"  История:")
print(f"    - Начальный год: {periods_config.get('history_start_year', 'auto')}")
print(f"    - Конечный год: {periods_config.get('history_end_year', 'auto')}")
print(f"  Прогноз:")
print(f"    - Начальный год: {periods_config.get('forecast_start_year', 'auto')}")
print(f"    - Конечный год: {periods_config.get('forecast_end_year', 'auto')}")
print(f"    - Количество лет: {periods_config.get('forecast_years', 'auto')}")

try:
    MODEL_VERSION = build_model(ROOT, COMPANY)
    print(f"📦 Версия модели: {MODEL_VERSION}")
    print(" ✅ Модель построена успешно!")
    mart = _open_data_mart()
    if mart is not None:
        versions = mart.get_existing_versions()
        if versions:
            is_forecast_dm = mart.get_model_results(MODEL_VERSION, 'IS', canonical=True)
            if not is_forecast_dm.empty:
                forecast_years_actual = [int(c) for c in is_forecast_dm.columns if str(c).isdigit()]
                forecast_years_actual = sorted(forecast_years_actual)
                if forecast_years_actual:
                    print(f"\n📊 Фактические годы прогноза: {forecast_years_actual}")
                    print(f"   Первый год: {forecast_years_actual[0]}")
                    print(f"   Последний год: {forecast_years_actual[-1]}")
                    print(f"   Всего лет: {len(forecast_years_actual)}")
        mart.close()
except Exception as e:
    print(f"\n❌ Ошибка построения модели: {e}")
    import traceback
    traceback.print_exc()
    raise



##3.1  Проверка баланса из БД после build_model

In [ ]:
# Проверка баланса из БД после build_model
from engine.database.data_mart import get_data_mart

mart = _open_data_mart()
if mart is not None:
    # Загружаем баланс из БД
    bs_df = mart.get_model_results(MODEL_VERSION, 'BS', canonical=True)
    
    print("\n" + "="*80)
    print("ПРОВЕРКА БАЛАНСА ИЗ БД:")
    print("="*80)
    
    # Проверяем баланс для прогнозных лет
    forecast_years = [2025, 2026, 2027, 2028, 2029]
    
    for year in forecast_years:
        year_str = str(year)
        if year_str in bs_df.columns:
            total_assets = bs_df.loc[bs_df['metric'] == 'total_assets', year_str].values
            total_liabilities = bs_df.loc[bs_df['metric'] == 'total_liabilities', year_str].values
            total_equity = bs_df.loc[bs_df['metric'] == 'total_equity', year_str].values
            
            if len(total_assets) > 0 and len(total_liabilities) > 0 and len(total_equity) > 0:
                ta = float(total_assets[0]) if not pd.isna(total_assets[0]) else 0.0
                tl = float(total_liabilities[0]) if not pd.isna(total_liabilities[0]) else 0.0
                te = float(total_equity[0]) if not pd.isna(total_equity[0]) else 0.0
                
                total_le = tl + te
                diff = ta - total_le
                
                print(f"\n{year}:")
                print(f"  Assets (БД): {ta:,.0f}")
                print(f"  Liabilities (БД): {tl:,.0f}")
                print(f"  Equity (БД): {te:,.0f}")
                print(f"  Liab+Equity (БД): {total_le:,.0f}")
                print(f"  Diff (БД): {diff:,.0f}")
            else:
                print(f"\n{year}: Данные не найдены в БД")
    
    mart.close()
else:
    print("⚠️ Не удалось открыть Data Mart")


## # Детальная диагностика баланса после build_model

In [ ]:
# Детальная диагностика баланса после build_modelfrom engine.model3.io import load_inputsfrom engine.model3.core import ThreeStatementModelfrom engine.model.preprocess import ModelPreprocessorfrom engine.database.data_mart import get_data_martimport yaml# Загружаем модель напрямую (как в build_model)mart = get_data_mart(ROOT, COMPANY)config = yaml.safe_load((croot / "configs" / "project.yaml").read_text(encoding='utf-8'))ModelPreprocessor(mart, config).run()mart.close()hist_state, _, drivers, _ = load_inputs(root=ROOT, company=COMPANY)forecast_years = sorted(y for y in drivers.revenue if y > hist_state.year)model = ThreeStatementModel(    hist_state,    forecast_years,    drivers,    zero_growth_test=False,    root=ROOT,    company=COMPANY)model.run()# Проверяем баланс напрямую из объекта моделиprint("\n" + "="*80)print("ПРЯМАЯ ПРОВЕРКА БАЛАНСА ИЗ ОБЪЕКТА МОДЕЛИ:")print("="*80)for year in forecast_years:    bs = model.BS[year]    total_assets = bs.get('total_assets', 0.0)    total_liabilities = bs.get('total_liabilities', 0.0)    total_equity = bs.get('total_equity', 0.0)    total_liab_equity = bs.get('total_liab_equity', 0.0)        calc_le = total_liabilities + total_equity    diff_assets_vs_le = total_assets - calc_le    diff_le_vs_total = calc_le - total_liab_equity        print(f"\n{year}:")    print(f"  Assets: {total_assets:,.0f}")    print(f"  Liabilities: {total_liabilities:,.0f}")    print(f"  Equity: {total_equity:,.0f}")    print(f"  Liab+Equity (calc): {calc_le:,.0f}")    print(f"  total_liab_equity (from BS): {total_liab_equity:,.0f}")    print(f"  Diff (Assets vs Liab+Equity): {diff_assets_vs_le:,.0f}")    print(f"  Diff (Liab+Equity vs total_liab_equity): {diff_le_vs_total:,.0f}")# Запускаем verificationprint("\n" + "="*80)print("РЕЗУЛЬТАТЫ VERIFICATION:")print("="*80)from engine.model3.verification import ModelVerifierverifier = ModelVerifier(model)results = verifier.verify_all()balance_checks = [r for r in results if r.check_name == "balance_identity"]for r in balance_checks:    if not r.passed:        print(f"\n{r.year}: diff={r.difference:,.2f}")        print(f"  Expected (Assets): {r.expected:,.0f}")        print(f"  Actual (Liab+Equity): {r.actual:,.0f}")        # Проверяем, что показывает прямая проверка        bs = model.BS[r.year]        total_assets = bs.get('total_assets', 0.0)        total_le = bs.get('total_liabilities', 0.0) + bs.get('total_equity', 0.0)        print(f"  Прямая проверка из BS: Assets={total_assets:,.0f}, Liab+Equity={total_le:,.0f}, diff={total_assets - total_le:,.0f}")

In [ ]:
# Детальный анализ сопоставимости истории и прогноза
# ВАЖНО: Эти функции определены один раз и используются во всем ноутбуке
# Не дублируйте их определения в других ячейках!



# Определение вспомогательных функций (определяются один раз)
def _open_data_mart():
    try:
        return get_data_mart(ROOT, COMPANY)
    except Exception as exc:
        print(f"⚠️ Не удалось подключиться к витрине данных: {exc}")
        return None

def load_history_from_db(statement_type='IS', canonical=False):
    stmt = statement_type.upper()
    df = pd.DataFrame()
    mart = _open_data_mart()
    if mart is not None:
        try:
            if stmt == 'IS':
                df = mart.get_history_income_statement(canonical=canonical)
            elif stmt == 'BS':
                df = mart.get_history_balance_sheet(canonical=canonical)
            else:
                df = mart.get_history_cash_flow(canonical=canonical)
            if df is not None and not df.empty:
                print(f"✅ История {stmt} загружена из витрины")
        except Exception as exc:
            print(f"⚠️ Не удалось загрузить историю {stmt} из витрины: {exc}")
        finally:
            mart.close()
    
    file_map = {
        'IS': f"history/is_history_{COMPANY}.csv",
        'BS': f"history/bs_history_{COMPANY}.csv",
        'CF': f"history/cf_history_{COMPANY}.csv"
    }
    
    if df is None or df.empty:
        csv_path = croot / file_map.get(stmt, '')
        if csv_path.exists():
            print(f"📄 История {stmt} загружена из CSV")
            return pd.read_csv(csv_path)
        print(f"❌ История {stmt} не найдена ни в витрине, ни в CSV")
        return pd.DataFrame()
    
    if not canonical and 'metric' in df.columns:
        year_cols = [c for c in df.columns if str(c).isdigit()]
        if year_cols:
            df_long = df.set_index('metric')[year_cols].T.reset_index().rename(columns={'index': 'year'})
            df_long['year'] = df_long['year'].astype(int)
            return df_long
    return df

def load_model_forecast_from_db(statement_type='IS', canonical=True):
    stmt = statement_type.upper()
    csv_filename = f"outputs/model/3statement_{stmt}.csv"
    return load_model_table(stmt, canonical=canonical, csv_filename=csv_filename)

def load_model_table(table_name: str, canonical: bool = False, csv_filename: Optional[str] = None):
    table_key = table_name.upper()
    df = pd.DataFrame()
    mart = _open_data_mart()
    if mart is not None:
        version = _resolve_model_version(mart)
        if version:
            try:
                df = mart.get_model_results(version, table_key, canonical=canonical)
                if df is not None and not df.empty:
                    print(f"✅ Таблица {table_key} загружена из витрины (версия {version})")
                    mart.close()
                    return df
            except Exception as exc:
                print(f"⚠️ Не удалось загрузить таблицу {table_key} из витрины: {exc}")
            finally:
                mart.close()
        else:
            mart.close()
    
    if csv_filename:
        csv_path = croot / csv_filename
        if csv_path.exists():
            print(f"⚠️ Используем legacy CSV для {table_key}: {csv_path.relative_to(ROOT)}")
            return pd.read_csv(csv_path)
    
    print(f"⚠️ Таблица {table_key} не найдена ни в витрине, ни в CSV")
    return pd.DataFrame()

def _resolve_model_version(mart):
    global MODEL_VERSION
    if MODEL_VERSION:
        return MODEL_VERSION
    try:
        versions = mart.get_existing_versions()
        if versions:
            MODEL_VERSION = versions[0]['version']
            return MODEL_VERSION
    except Exception as exc:
        print(f"⚠️ Не удалось определить версию модели: {exc}")
    return None

# Проверка переменных окружения
if 'ROOT' not in globals():
    current_dir = Path.cwd()
    if (current_dir / 'engine').exists():
        ROOT = current_dir
    elif (current_dir.parent / 'engine').exists():
        ROOT = current_dir.parent
    elif (current_dir.parent.parent / 'engine').exists():
        ROOT = current_dir.parent.parent
    else:
        nb_path = Path(__file__) if '__file__' in globals() else Path.cwd()
        ROOT = nb_path.parent.parent.parent
    print(f"ROOT: {ROOT}")

if 'COMPANY' not in globals():
    COMPANY = 'us_steel'  # fallback
    current_dir = Path.cwd()
    if 'companies' in current_dir.parts:
        companies_idx = current_dir.parts.index('companies')
        if companies_idx + 1 < len(current_dir.parts):
            COMPANY = current_dir.parts[companies_idx + 1]
    print(f"COMPANY: {COMPANY}")

if 'croot' not in globals():
    croot = ROOT / 'companies' / COMPANY
    print(f"Company root: {croot}")

if 'MODEL_VERSION' not in globals():
    MODEL_VERSION = None

display(Markdown("### 🔍 Детальный анализ каждой статьи: История vs Прогноз"))

# Детальный анализ сопоставимости истории и прогноза


# Загружаем данные если еще не загружены
if 'hist_is' not in locals() or hist_is.empty:
    hist_is = load_history_from_db('IS', canonical=True)
if 'hist_bs' not in locals() or hist_bs.empty:
    hist_bs = load_history_from_db('BS', canonical=True)
if 'hist_cf' not in locals() or hist_cf.empty:
    hist_cf = load_history_from_db('CF', canonical=True)

if 'forecast_is' not in locals() or forecast_is.empty:
    forecast_is = load_model_forecast_from_db('IS', canonical=True)
if 'forecast_bs' not in locals() or forecast_bs.empty:
    forecast_bs = load_model_forecast_from_db('BS', canonical=True)
if 'forecast_cf' not in locals() or forecast_cf.empty:
    forecast_cf = load_model_forecast_from_db('CF', canonical=True)

# Преобразуем в wide format
def to_wide(df, metric_col='metric'):
    if df.empty:
        return pd.DataFrame()
    if 'year' in df.columns and metric_col in df.columns:
        return df.pivot(index=metric_col, columns='year', values='value')
    elif metric_col in df.columns:
        return df.set_index(metric_col)
    return df

hist_is_wide = to_wide(hist_is)
hist_bs_wide = to_wide(hist_bs)
hist_cf_wide = to_wide(hist_cf)
forecast_is_wide = to_wide(forecast_is)
forecast_bs_wide = to_wide(forecast_bs)
forecast_cf_wide = to_wide(forecast_cf)

# Получаем годы
hist_years = sorted([int(c) for c in hist_is_wide.columns if str(c).isdigit() and int(c) <= 2024])[-5:]
forecast_years = sorted([int(c) for c in forecast_is_wide.columns if str(c).isdigit() and int(c) > 2024])

def analyze_item(item_name, hist_df, forecast_df, statement_type='IS'):
    """Анализирует одну статью, сравнивая историю и прогноз"""
    issues = []
    warnings = []
    
    # Проверяем наличие в истории и прогнозе
    in_hist = item_name in hist_df.index if not hist_df.empty else False
    in_forecast = item_name in forecast_df.index if not forecast_df.empty else False
    
    if not in_hist and not in_forecast:
        return None
    
    # Получаем значения
    hist_values = {}
    forecast_values = {}
    
    if in_hist:
        for y in hist_years:
            val = pd.to_numeric(hist_df.loc[item_name, str(y)], errors='coerce') if str(y) in hist_df.columns else None
            if pd.notna(val):
                hist_values[y] = float(val)
    
    if in_forecast:
        for y in forecast_years:
            val = pd.to_numeric(forecast_df.loc[item_name, str(y)], errors='coerce') if str(y) in forecast_df.columns else None
            if pd.notna(val):
                forecast_values[y] = float(val)
    
    if not hist_values and not forecast_values:
        return None
    
    # Анализ
    result = {
        'Статья': item_name,
        'В истории': '✅' if in_hist else '❌',
        'В прогнозе': '✅' if in_forecast else '❌',
        'Проблемы': [],
        'Предупреждения': []
    }
    
    if hist_values and forecast_values:
        # 1. Проверка знаков
        hist_signs = [1 if v > 0 else (-1 if v < 0 else 0) for v in hist_values.values()]
        forecast_signs = [1 if v > 0 else (-1 if v < 0 else 0) for v in forecast_values.values()]
        
        hist_typical_sign = max(set(hist_signs), key=hist_signs.count) if hist_signs else 0
        forecast_typical_sign = max(set(forecast_signs), key=forecast_signs.count) if forecast_signs else 0
        
        if hist_typical_sign != 0 and forecast_typical_sign != 0 and hist_typical_sign != forecast_typical_sign:
            result['Проблемы'].append(f"❌ Смена знака: история {('+' if hist_typical_sign > 0 else '-')}, прогноз {('+' if forecast_typical_sign > 0 else '-')}")
        
        # 2. Проверка порядка чисел
        hist_abs_values = [abs(v) for v in hist_values.values() if v != 0]
        forecast_abs_values = [abs(v) for v in forecast_values.values() if v != 0]
        
        if hist_abs_values and forecast_abs_values:
            hist_avg_magnitude = np.mean(hist_abs_values)
            forecast_avg_magnitude = np.mean(forecast_abs_values)
            
            # Проверяем разницу в порядке
            if hist_avg_magnitude > 0:
                ratio = forecast_avg_magnitude / hist_avg_magnitude
                if ratio > 10 or ratio < 0.1:
                    result['Проблемы'].append(f"❌ Разница в масштабе: {ratio:.1f}x (история ~${hist_avg_magnitude:,.0f}, прогноз ~${forecast_avg_magnitude:,.0f})")
                elif ratio > 3 or ratio < 0.33:
                    result['Предупреждения'].append(f"⚠️ Значительное изменение масштаба: {ratio:.1f}x")
        
        # 3. Проверка тренда
        if len(hist_values) >= 2 and len(forecast_values) >= 1:
            hist_last = list(hist_values.values())[-1]
            hist_prev = list(hist_values.values())[-2]
            forecast_first = list(forecast_values.values())[0]
            
            hist_change = hist_last - hist_prev
            forecast_jump = forecast_first - hist_last
            
            # Проверяем резкий скачок
            if abs(hist_change) > 0:
                jump_ratio = abs(forecast_jump) / abs(hist_change) if hist_change != 0 else float('inf')
                if jump_ratio > 5:
                    result['Проблемы'].append(f"❌ Резкий скачок на границе: изменение {forecast_jump:,.0f} vs историческое {hist_change:,.0f} ({jump_ratio:.1f}x)")
    
    elif not in_hist and in_forecast:
        result['Предупреждения'].append(f"⚠️ Статья отсутствует в истории, но есть в прогнозе")
    elif in_hist and not in_forecast:
        result['Предупреждения'].append(f"⚠️ Статья есть в истории, но отсутствует в прогнозе")
    
    return result

# Анализируем все статьи
all_items = set()
if not hist_is_wide.empty:
    all_items.update(hist_is_wide.index)
if not forecast_is_wide.empty:
    all_items.update(forecast_is_wide.index)
if not hist_bs_wide.empty:
    all_items.update(hist_bs_wide.index)
if not forecast_bs_wide.empty:
    all_items.update(forecast_bs_wide.index)
if not hist_cf_wide.empty:
    all_items.update(hist_cf_wide.index)
if not forecast_cf_wide.empty:
    all_items.update(forecast_cf_wide.index)

# Анализ IS
is_analysis = []
is_items = ['revenue', 'cogs', 'sga', 'depreciation_owned', 'depreciation_rou', 'amortization', 
            'ebitda', 'ebit', 'interest_income', 'interest_expense', 'lease_interest', 'tax', 'net_income']
for item in is_items:
    result = analyze_item(item, hist_is_wide, forecast_is_wide, 'IS')
    if result:
        is_analysis.append(result)

# Анализ BS
bs_analysis = []
bs_items = ['cash', 'ar', 'inventory', 'ppe', 'intangibles', 'rou_asset', 'total_assets',
            'st_debt', 'ap', 'debt', 'lease_liability', 'equity', 'total_liabilities', 'total_equity']
for item in bs_items:
    result = analyze_item(item, hist_bs_wide, forecast_bs_wide, 'BS')
    if result:
        bs_analysis.append(result)

# Анализ CF
cf_analysis = []
cf_items = ['net_income', 'depreciation', 'amortization', 'wc_delta', 'cfo', 'capex', 'cfi', 
            'debt_borrowings', 'debt_repayments', 'dividends', 'cff', 'net_change']
for item in cf_items:
    result = analyze_item(item, hist_cf_wide, forecast_cf_wide, 'CF')
    if result:
        cf_analysis.append(result)

# Выводим результаты
def print_analysis(analysis_list, title):
    
    for item in analysis_list:
        display(Markdown(f"#### {item['Статья']}"))
        print(f"  В истории: {item['В истории']} | В прогнозе: {item['В прогнозе']}")
        
        if item['Проблемы']:
            print(f"  \n  🔴 ПРОБЛЕМЫ:")
            for prob in item['Проблемы']:
                print(f"    {prob}")
        
        if item['Предупреждения']:
            print(f"  \n  ⚠️ ПРЕДУПРЕЖДЕНИЯ:")
            for warn in item['Предупреждения']:
                print(f"    {warn}")
        
        if not item['Проблемы'] and not item['Предупреждения']:
            print(f"  ✅ Нет проблем")
        
        print()

print_analysis(is_analysis, '📊 Income Statement (IS)')
print_analysis(bs_analysis, '🏦 Balance Sheet (BS)')
print_analysis(cf_analysis, '💰 Cash Flow Statement (CF)')

# Сводная таблица проблем
all_issues = []
for analysis in [is_analysis, bs_analysis, cf_analysis]:
    for item in analysis:
        if item['Проблемы'] or item['Предупреждения']:
            all_issues.append({
                'Статья': item['Статья'],
                'Проблем': len(item['Проблемы']),
                'Предупреждений': len(item['Предупреждения']),
                'Статус': '❌' if item['Проблемы'] else '⚠️'
            })

if all_issues:
    display(Markdown("### 📋 Сводная таблица проблем"))
    issues_df = pd.DataFrame(all_issues)
    display(issues_df)
    print(f"\nВсего статей с проблемами: {len(all_issues)}")
    print(f"Критических проблем: {sum(1 for i in all_issues if i['Проблем'] > 0)}")
    print(f"Предупреждений: {sum(1 for i in all_issues if i['Предупреждений'] > 0)}")
else:
    print("✅ Все статьи проверены, критических проблем не обнаружено")

## ✅ Комплексное тестирование модели: Полнота, Корректность, Логика <a id="section-testing"></a>

Этот раздел содержит систематические тесты для проверки:
- ✅ Полнота данных: все метрики присутствуют в прогнозе
- ✅ Корректность corkscrews: PP&E, Intangibles, Debt, WC, Tax, Lease
- ✅ Финансовые связи: BS Identity, Cash Bridge, Retained Earnings
- ✅ Соответствие историческим трендам
- ✅ Наличие драйверов для всех статей
- ✅ Логика моделирования работает корректно
- ✅ Контрольные точки пройдены


In [ ]:
display(Markdown("### ✅ Тест 1: Полнота данных - проверка наличия всех метрик в прогнозе"))

# Загружаем прогнозы
forecast_is = load_model_forecast_from_db('IS', canonical=True)
forecast_bs = load_model_forecast_from_db('BS', canonical=True)
forecast_cf = load_model_forecast_from_db('CF', canonical=True)

# Критические метрики, которые должны быть в прогнозе
critical_metrics = {
    'IS': ['revenue', 'cogs', 'sga', 'ebitda', 'ebit', 'depreciation_owned', 'amortization', 
           'interest_expense', 'interest_income', 'tax_expense', 'net_income'],
    'BS': ['cash', 'accounts_receivable', 'inventory', 'ppe_net', 'ppe_gross', 'ppe_accum_dep',
           'intangibles', 'total_assets', 'accounts_payable', 'short_term_debt', 'long_term_debt',
           'total_liabilities', 'retained_earnings', 'total_equity', 'total_liab_equity'],
    'CF': ['cfo', 'cfi', 'cff', 'net_change', 'capex', 'depreciation', 'amortization', 'wc_delta']
}

completeness_results = []

for stmt_type, metrics in critical_metrics.items():
    forecast_df = forecast_is if stmt_type == 'IS' else (forecast_bs if stmt_type == 'BS' else forecast_cf)
    
    if forecast_df.empty:
        completeness_results.append({
            'Отчет': stmt_type,
            'Статус': '❌',
            'Детали': 'Прогноз отсутствует'
        })
        continue
    
    # Преобразуем в wide format если нужно
    if 'year' in forecast_df.columns and 'metric' in forecast_df.columns:
        forecast_wide = forecast_df.pivot(index='metric', columns='year', values='value')
    else:
        forecast_wide = forecast_df.set_index('metric') if 'metric' in forecast_df.columns else forecast_df
    
    forecast_year_cols = [c for c in forecast_wide.columns if str(c).isdigit() and int(c) > 2024]
    
    if not forecast_year_cols:
        completeness_results.append({
            'Отчет': stmt_type,
            'Статус': '❌',
            'Детали': 'Нет прогнозных лет'
        })
        continue
    
    missing_metrics = []
    partial_metrics = []
    complete_metrics = []
    
    for metric in metrics:
        if metric not in forecast_wide.index:
            missing_metrics.append(metric)
        else:
            # Проверяем наличие значений для всех прогнозных лет
            values_count = 0
            for y in forecast_year_cols:
                val = pd.to_numeric(forecast_wide.loc[metric, str(y)], errors='coerce')
                if pd.notna(val) and val != 0:
                    values_count += 1
            
            if values_count == 0:
                missing_metrics.append(metric)
            elif values_count < len(forecast_year_cols):
                partial_metrics.append(f"{metric} ({values_count}/{len(forecast_year_cols)} лет)")
            else:
                complete_metrics.append(metric)
    
    status = '✅' if not missing_metrics and not partial_metrics else ('⚠️' if partial_metrics else '❌')
    details = f"Полных: {len(complete_metrics)}, Частичных: {len(partial_metrics)}, Отсутствует: {len(missing_metrics)}"
    if missing_metrics:
        details += f" | Отсутствуют: {', '.join(missing_metrics[:5])}"
    
    completeness_results.append({
        'Отчет': stmt_type,
        'Статус': status,
        'Детали': details
    })

completeness_df = pd.DataFrame(completeness_results)
display(completeness_df)

# Детальная таблица по метрикам
display(Markdown("#### 📊 Детальная проверка метрик"))
detailed_check = []
for stmt_type, metrics in critical_metrics.items():
    forecast_df = forecast_is if stmt_type == 'IS' else (forecast_bs if stmt_type == 'BS' else forecast_cf)
    if forecast_df.empty:
        continue
    
    if 'year' in forecast_df.columns and 'metric' in forecast_df.columns:
        forecast_wide = forecast_df.pivot(index='metric', columns='year', values='value')
    else:
        forecast_wide = forecast_df.set_index('metric') if 'metric' in forecast_df.columns else forecast_df
    
    forecast_year_cols = sorted([int(c) for c in forecast_wide.columns if str(c).isdigit() and int(c) > 2024])
    
    for metric in metrics:
        row = {'Отчет': stmt_type, 'Метрика': metric}
        if metric in forecast_wide.index:
            for y in forecast_year_cols[:3]:  # Первые 3 года прогноза
                val = pd.to_numeric(forecast_wide.loc[metric, str(y)], errors='coerce')
                row[f'{y}'] = '✅' if pd.notna(val) else '❌'
        else:
            for y in forecast_year_cols[:3]:
                row[f'{y}'] = '❌'
        detailed_check.append(row)

if detailed_check:
    detailed_df = pd.DataFrame(detailed_check)
    display(detailed_df.head(30))  # Показываем первые 30 строк


In [ ]:
display(Markdown("### ✅ Тест 2: Корректность Corkscrews - проверка формул"))

corkscrew_tests = []

# 1. PP&E Corkscrew: PP&E_Net(t) = PP&E_Gross(t) - PP&E_AccumDep(t)
if not forecast_bs.empty:
    if 'year' in forecast_bs.columns and 'metric' in forecast_bs.columns:
        bs_wide = forecast_bs.pivot(index='metric', columns='year', values='value')
    else:
        bs_wide = forecast_bs.set_index('metric') if 'metric' in forecast_bs.columns else forecast_bs
    
    forecast_year_cols = sorted([int(c) for c in bs_wide.columns if str(c).isdigit() and int(c) > 2024])
    
    if 'ppe_net' in bs_wide.index and 'ppe_gross' in bs_wide.index and 'ppe_accum_dep' in bs_wide.index:
        ppe_errors = []
        for y in forecast_year_cols[:5]:
            y_str = str(y)
            ppe_net = pd.to_numeric(bs_wide.loc['ppe_net', y_str], errors='coerce')
            ppe_gross = pd.to_numeric(bs_wide.loc['ppe_gross', y_str], errors='coerce')
            ppe_accum = pd.to_numeric(bs_wide.loc['ppe_accum_dep', y_str], errors='coerce')
            
            if all(pd.notna(v) for v in [ppe_net, ppe_gross, ppe_accum]):
                calculated_net = ppe_gross - ppe_accum
                diff = abs(ppe_net - calculated_net)
                if diff > 1.0:  # Допустимая погрешность $1M
                    ppe_errors.append(f"{y}: diff=${diff:,.0f}")
        
        corkscrew_tests.append({
            'Corkscrew': 'PP&E',
            'Формула': 'PP&E_Net = PP&E_Gross - PP&E_AccumDep',
            'Статус': '✅' if not ppe_errors else '❌',
            'Ошибки': ', '.join(ppe_errors) if ppe_errors else 'Нет'
        })
    
    # 2. PP&E Corkscrew: PP&E_Gross(t) = PP&E_Gross(t-1) + CapEx - Disposals
    if 'ppe_gross' in bs_wide.index:
        ppe_gross_errors = []
        capex_df = load_model_table('CAPEX_BREAKDOWN', csv_filename='outputs/model/capex_breakdown.csv')
        
        if not capex_df.empty:
            if 'year' in capex_df.columns and 'metric' in capex_df.columns:
                capex_wide = capex_df.pivot(index='metric', columns='year', values='value')
            else:
                capex_wide = capex_df.set_index('metric') if 'metric' in capex_df.columns else capex_df
            
            # Загружаем историю для первого года
            hist_bs_temp = load_history_from_db('BS', canonical=True)
            if not hist_bs_temp.empty:
                if 'year' in hist_bs_temp.columns and 'metric' in hist_bs_temp.columns:
                    hist_bs_wide_temp = hist_bs_temp.pivot(index='metric', columns='year', values='value')
                else:
                    hist_bs_wide_temp = hist_bs_temp.set_index('metric') if 'metric' in hist_bs_temp.columns else hist_bs_temp
                
                history_end_year = max([int(c) for c in hist_bs_wide_temp.columns if str(c).isdigit() and int(c) <= 2024])
                prev_ppe_gross = pd.to_numeric(hist_bs_wide_temp.loc['ppe_gross', str(history_end_year)], errors='coerce') if 'ppe_gross' in hist_bs_wide_temp.index else None
                
                for i, y in enumerate(forecast_year_cols[:5]):
                    y_str = str(y)
                    curr_ppe_gross = pd.to_numeric(bs_wide.loc['ppe_gross', y_str], errors='coerce')
                    total_capex = pd.to_numeric(capex_wide.loc['total_capex', y_str], errors='coerce') if 'total_capex' in capex_wide.index else None
                    disposals = pd.to_numeric(capex_wide.loc['disposals', y_str], errors='coerce') if 'disposals' in capex_wide.index else 0.0
                    
                    if pd.notna(curr_ppe_gross) and pd.notna(prev_ppe_gross) and pd.notna(total_capex):
                        calculated_gross = prev_ppe_gross + total_capex - (disposals or 0.0)
                        diff = abs(curr_ppe_gross - calculated_gross)
                        if diff > 10.0:  # Допустимая погрешность $10M
                            ppe_gross_errors.append(f"{y}: diff=${diff:,.0f}")
                    
                    prev_ppe_gross = curr_ppe_gross if pd.notna(curr_ppe_gross) else prev_ppe_gross
        
        corkscrew_tests.append({
            'Corkscrew': 'PP&E Gross Roll',
            'Формула': 'PP&E_Gross(t) = PP&E_Gross(t-1) + CapEx - Disposals',
            'Статус': '✅' if not ppe_gross_errors else '⚠️',
            'Ошибки': ', '.join(ppe_gross_errors) if ppe_gross_errors else 'Нет'
        })
    
    # 3. Retained Earnings: RE(t) = RE(t-1) + Net Income - Dividends
    if 'retained_earnings' in bs_wide.index:
        # Загружаем forecast_is и forecast_cf если еще не загружены
        if 'forecast_is' not in locals() or forecast_is.empty:
            forecast_is = load_model_forecast_from_db('IS', canonical=True)
        if 'forecast_cf' not in locals() or forecast_cf.empty:
            forecast_cf = load_model_forecast_from_db('CF', canonical=True)
        
        if not forecast_is.empty:
            if 'year' in forecast_is.columns and 'metric' in forecast_is.columns:
                is_wide = forecast_is.pivot(index='metric', columns='year', values='value')
            else:
                is_wide = forecast_is.set_index('metric') if 'metric' in forecast_is.columns else forecast_is
            
            if not forecast_cf.empty:
                if 'year' in forecast_cf.columns and 'metric' in forecast_cf.columns:
                    cf_wide = forecast_cf.pivot(index='metric', columns='year', values='value')
                else:
                    cf_wide = forecast_cf.set_index('metric') if 'metric' in forecast_cf.columns else forecast_cf
            else:
                cf_wide = pd.DataFrame()
        
        if 'is_wide' in locals() and 'net_income' in is_wide.index:
            re_errors = []
            # Загружаем историю для первого года
            hist_bs_temp = load_history_from_db('BS', canonical=True)
            if not hist_bs_temp.empty:
                if 'year' in hist_bs_temp.columns and 'metric' in hist_bs_temp.columns:
                    hist_bs_wide_temp = hist_bs_temp.pivot(index='metric', columns='year', values='value')
                else:
                    hist_bs_wide_temp = hist_bs_temp.set_index('metric') if 'metric' in hist_bs_temp.columns else hist_bs_temp
                
                history_end_year = max([int(c) for c in hist_bs_wide_temp.columns if str(c).isdigit() and int(c) <= 2024])
                prev_re = pd.to_numeric(hist_bs_wide_temp.loc['retained_earnings', str(history_end_year)], errors='coerce') if 'retained_earnings' in hist_bs_wide_temp.index else None
                
                for i, y in enumerate(forecast_year_cols[:5]):
                    y_str = str(y)
                    curr_re = pd.to_numeric(bs_wide.loc['retained_earnings', y_str], errors='coerce')
                    ni = pd.to_numeric(is_wide.loc['net_income', y_str], errors='coerce')
                    dividends = pd.to_numeric(forecast_cf_wide.loc['cff_dividends', y_str], errors='coerce') if 'cff_dividends' in forecast_cf_wide.index else 0.0
                    
                    if pd.notna(curr_re) and pd.notna(prev_re) and pd.notna(ni):
                        calculated_re = prev_re + ni - (dividends or 0.0)
                        diff = abs(curr_re - calculated_re)
                        if diff > 1.0:
                            re_errors.append(f"{y}: diff=${diff:,.0f}")
                    
                    prev_re = curr_re if pd.notna(curr_re) else prev_re
        
        corkscrew_tests.append({
            'Corkscrew': 'Retained Earnings',
            'Формула': 'RE(t) = RE(t-1) + Net Income - Dividends',
            'Статус': '✅' if not re_errors else '❌',
            'Ошибки': ', '.join(re_errors) if re_errors else 'Нет'
        })

if corkscrew_tests:
    corkscrew_df = pd.DataFrame(corkscrew_tests)
    display(corkscrew_df)
else:
    print("⚠️ Не удалось проверить corkscrews - данные отсутствуют")


In [ ]:
display(Markdown("### ✅ Тест 3: Финансовые связи - BS Identity, Cash Bridge, Retained Earnings"))

financial_checks = []

# 1. BS Identity: Total Assets = Total Liabilities + Equity
if not forecast_bs.empty:
    if 'year' in forecast_bs.columns and 'metric' in forecast_bs.columns:
        bs_wide = forecast_bs.pivot(index='metric', columns='year', values='value')
    else:
        bs_wide = forecast_bs.set_index('metric') if 'metric' in forecast_bs.columns else forecast_bs
    
    forecast_year_cols = sorted([int(c) for c in bs_wide.columns if str(c).isdigit() and int(c) > 2024])
    
    bs_identity_errors = []
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        if 'total_assets' in bs_wide.index and 'total_liab_equity' in bs_wide.index:
            assets = pd.to_numeric(bs_wide.loc['total_assets', y_str], errors='coerce')
            liab_equity = pd.to_numeric(bs_wide.loc['total_liab_equity', y_str], errors='coerce')
            
            if pd.notna(assets) and pd.notna(liab_equity):
                diff = abs(assets - liab_equity)
                diff_pct = (diff / assets * 100) if assets != 0 else 0
                if diff > 1.0:  # Допустимая погрешность $1M
                    bs_identity_errors.append(f"{y}: diff=${diff:,.0f} ({diff_pct:.4f}%)")
    
    financial_checks.append({
        'Проверка': 'BS Identity',
        'Формула': 'Total Assets = Total Liabilities + Equity',
        'Статус': '✅' if not bs_identity_errors else '❌',
        'Ошибки': ', '.join(bs_identity_errors) if bs_identity_errors else 'Нет',
        'Годы проверки': len(forecast_year_cols[:5])
    })

# 2. Cash Bridge: ΔCash(BS) = NetChange(CF)
if not forecast_bs.empty and not forecast_cf.empty:
    if 'year' in forecast_cf.columns and 'metric' in forecast_cf.columns:
        cf_wide = forecast_cf.pivot(index='metric', columns='year', values='value')
    else:
        cf_wide = forecast_cf.set_index('metric') if 'metric' in forecast_cf.columns else forecast_cf
    
    cash_bridge_errors = []
    
    # Загружаем историю для первого года
    hist_bs_temp = load_history_from_db('BS', canonical=True)
    if not hist_bs_temp.empty:
        if 'year' in hist_bs_temp.columns and 'metric' in hist_bs_temp.columns:
            hist_bs_wide_temp = hist_bs_temp.pivot(index='metric', columns='year', values='value')
        else:
            hist_bs_wide_temp = hist_bs_temp.set_index('metric') if 'metric' in hist_bs_temp.columns else hist_bs_temp
        
        history_end_year = max([int(c) for c in hist_bs_wide_temp.columns if str(c).isdigit() and int(c) <= 2024])
        prev_cash = pd.to_numeric(hist_bs_wide_temp.loc['cash', str(history_end_year)], errors='coerce') if 'cash' in hist_bs_wide_temp.index else None
        
        for y in forecast_year_cols[:5]:
            y_str = str(y)
            curr_cash = pd.to_numeric(bs_wide.loc['cash', y_str], errors='coerce')
            net_change = pd.to_numeric(cf_wide.loc['net_change', y_str], errors='coerce') if 'net_change' in cf_wide.index else None
            
            if pd.notna(curr_cash) and pd.notna(prev_cash) and pd.notna(net_change):
                cash_delta = curr_cash - prev_cash
                diff = abs(cash_delta - net_change)
                if diff > 1.0:
                    cash_bridge_errors.append(f"{y}: diff=${diff:,.0f} (ΔCash=${cash_delta:,.0f}, NetChange=${net_change:,.0f})")
            
            prev_cash = curr_cash if pd.notna(curr_cash) else prev_cash
    
    financial_checks.append({
        'Проверка': 'Cash Bridge',
        'Формула': 'ΔCash(BS) = NetChange(CF)',
        'Статус': '✅' if not cash_bridge_errors else '❌',
        'Ошибки': ', '.join(cash_bridge_errors) if cash_bridge_errors else 'Нет',
        'Годы проверки': len(forecast_year_cols[:5])
    })

# 3. CFO Formula: CFO = Net Income + D&A - WC Delta - Tax Paid (упрощенная проверка)
# Убеждаемся что is_wide и cf_wide определены
if 'forecast_is' not in locals() or forecast_is.empty:
    forecast_is = load_model_forecast_from_db('IS', canonical=True)
if 'forecast_cf' not in locals() or forecast_cf.empty:
    forecast_cf = load_model_forecast_from_db('CF', canonical=True)

if not forecast_is.empty and not forecast_cf.empty:
    if 'is_wide' not in locals():
        if 'year' in forecast_is.columns and 'metric' in forecast_is.columns:
            is_wide = forecast_is.pivot(index='metric', columns='year', values='value')
        else:
            is_wide = forecast_is.set_index('metric') if 'metric' in forecast_is.columns else forecast_is
    
    if 'cf_wide' not in locals():
        if 'year' in forecast_cf.columns and 'metric' in forecast_cf.columns:
            cf_wide = forecast_cf.pivot(index='metric', columns='year', values='value')
        else:
            cf_wide = forecast_cf.set_index('metric') if 'metric' in forecast_cf.columns else forecast_cf
    cfo_formula_errors = []
    
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        ni = pd.to_numeric(is_wide.loc['net_income', y_str], errors='coerce') if 'net_income' in is_wide.index else None
        dep = pd.to_numeric(is_wide.loc['depreciation_owned', y_str], errors='coerce') if 'depreciation_owned' in is_wide.index else 0.0
        amort = pd.to_numeric(is_wide.loc['amortization', y_str], errors='coerce') if 'amortization' in is_wide.index else 0.0
        wc_delta = pd.to_numeric(cf_wide.loc['wc_delta', y_str], errors='coerce') if 'wc_delta' in cf_wide.index else 0.0
        tax_paid = pd.to_numeric(cf_wide.loc['tax_paid', y_str], errors='coerce') if 'tax_paid' in cf_wide.index else 0.0
        cfo = pd.to_numeric(cf_wide.loc['cfo', y_str], errors='coerce') if 'cfo' in cf_wide.index else None
        
        if all(pd.notna(v) for v in [ni, cfo] if v is not None):
            da = (dep or 0.0) + (amort or 0.0)
            calculated_cfo = ni + da - (wc_delta or 0.0) - (tax_paid or 0.0)
            diff = abs(cfo - calculated_cfo)
            if diff > 10.0:  # Допустимая погрешность $10M (учитывая другие компоненты CFO)
                cfo_formula_errors.append(f"{y}: diff=${diff:,.0f}")
    
    financial_checks.append({
        'Проверка': 'CFO Formula',
        'Формула': 'CFO ≈ Net Income + D&A - WC Delta - Tax Paid',
        'Статус': '✅' if not cfo_formula_errors else '⚠️',
        'Ошибки': ', '.join(cfo_formula_errors) if cfo_formula_errors else 'Нет',
        'Годы проверки': len(forecast_year_cols[:5])
    })

if financial_checks:
    financial_df = pd.DataFrame(financial_checks)
    display(financial_df)
else:
    print("⚠️ Не удалось проверить финансовые связи - данные отсутствуют")


In [ ]:
display(Markdown("### ✅ Тест 4: Соответствие историческим трендам"))

# Проверяем, что прогноз не слишком сильно отклоняется от исторических трендов
trend_tests = []

# Загружаем историю если еще не загружена
if 'hist_is' not in locals() or hist_is.empty:
    hist_is = load_history_from_db('IS', canonical=True)
if 'forecast_is' not in locals() or forecast_is.empty:
    forecast_is = load_model_forecast_from_db('IS', canonical=True)

if not hist_is.empty and not forecast_is.empty:
    # Преобразуем в wide format
    if 'year' in hist_is.columns and 'metric' in hist_is.columns:
        hist_is_wide = hist_is.pivot(index='metric', columns='year', values='value')
    else:
        hist_is_wide = hist_is.set_index('metric') if 'metric' in hist_is.columns else hist_is
    
    if 'year' in forecast_is.columns and 'metric' in forecast_is.columns:
        forecast_is_wide = forecast_is.pivot(index='metric', columns='year', values='value')
    else:
        forecast_is_wide = forecast_is.set_index('metric') if 'metric' in forecast_is.columns else forecast_is
    
    hist_year_cols = sorted([int(c) for c in hist_is_wide.columns if str(c).isdigit() and int(c) <= 2024])
    forecast_year_cols = sorted([int(c) for c in forecast_is_wide.columns if str(c).isdigit() and int(c) > 2024])
    
    # Ключевые метрики для проверки трендов
    trend_metrics = ['revenue', 'cogs', 'sga', 'ebitda', 'net_income']
    
    for metric in trend_metrics:
        if metric not in hist_is_wide.index or metric not in forecast_is_wide.index:
            continue
        
        # Берем последние 3 года истории
        last_3_years = hist_year_cols[-3:] if len(hist_year_cols) >= 3 else hist_year_cols
        first_forecast_year = forecast_year_cols[0] if forecast_year_cols else None
        
        if not last_3_years or not first_forecast_year:
            continue
        
        # Рассчитываем средний темп роста за последние 3 года
        hist_values = [pd.to_numeric(hist_is_wide.loc[metric, str(y)], errors='coerce') for y in last_3_years]
        hist_values = [v for v in hist_values if pd.notna(v) and v > 0]
        
        if len(hist_values) < 2:
            continue
        
        # CAGR за последние 3 года
        growth_rates = []
        for i in range(1, len(hist_values)):
            if hist_values[i-1] > 0:
                growth = (hist_values[i] / hist_values[i-1]) - 1.0
                growth_rates.append(growth)
        
        avg_growth = np.mean(growth_rates) if growth_rates else 0.0
        
        # Прогнозное значение для первого года
        last_hist_value = hist_values[-1]
        forecast_value = pd.to_numeric(forecast_is_wide.loc[metric, str(first_forecast_year)], errors='coerce')
        
        if pd.notna(forecast_value) and last_hist_value > 0:
            forecast_growth = (forecast_value / last_hist_value) - 1.0
            
            # Проверяем, что прогнозный рост не слишком сильно отклоняется от исторического
            # Допускаем отклонение до 50% от исторического тренда
            max_deviation = abs(avg_growth) * 1.5 if avg_growth != 0 else 1.0
            deviation = abs(forecast_growth - avg_growth)
            
            status = '✅' if deviation <= max_deviation else '⚠️'
            
            trend_tests.append({
                'Метрика': metric.replace('_', ' ').title(),
                'Исторический CAGR (3 года)': f"{avg_growth:.1%}",
                'Прогнозный рост (1-й год)': f"{forecast_growth:.1%}",
                'Отклонение': f"{deviation:.1%}",
                'Статус': status,
                'Примечание': 'Соответствует тренду' if deviation <= max_deviation else 'Отклонение от тренда'
            })

if trend_tests:
    trend_df = pd.DataFrame(trend_tests)
    display(trend_df)
else:
    print("⚠️ Не удалось проверить соответствие трендам - недостаточно данных")


In [ ]:
display(Markdown("### ✅ Тест 5: Наличие драйверов и логики моделирования"))

# Проверяем, что для всех статей есть драйверы или логика моделирования
driver_tests = []

# Загружаем конфигурацию проекта
proj_yaml_path = croot / 'configs' / 'project.yaml'
if proj_yaml_path.exists():
    with open(proj_yaml_path, 'r', encoding='utf-8') as f:
        proj_yaml = yaml.safe_load(f)
    
    model_config = proj_yaml.get('model', {}).get('standard', {})
    
    # Проверяем наличие драйверов для ключевых статей
    driver_checks = {
        'Revenue': {
            'driver': 'macro_forecast' if proj_yaml.get('macro_forecast', {}).get('factors') else None,
            'method': 'Регрессия на макрофакторы' if proj_yaml.get('macro_forecast', {}).get('factors') else 'EWA темп роста',
            'required': True
        },
        'COGS': {
            'driver': model_config.get('cogs', {}).get('use_ppi_uplift', False),
            'method': f"% от Revenue × PPI uplift" if model_config.get('cogs', {}).get('use_ppi_uplift') else '% от Revenue',
            'required': True
        },
        'SG&A': {
            'driver': model_config.get('sgna', {}).get('index_by_cpi', False),
            'method': f"EWA темп + CPI индексация" if model_config.get('sgna', {}).get('index_by_cpi') else 'EWA темп роста',
            'required': True
        },
        'CapEx': {
            'driver': model_config.get('capex', {}).get('method', 'ratio_to_revenue'),
            'method': f"Метод: {model_config.get('capex', {}).get('method', 'ratio_to_revenue')}",
            'required': True
        },
        'Depreciation': {
            'driver': proj_yaml.get('features', {}).get('use_ppe_corkscrew', False),
            'method': 'PP&E corkscrew' if proj_yaml.get('features', {}).get('use_ppe_corkscrew') else '% метод',
            'required': True
        },
        'Working Capital': {
            'driver': proj_yaml.get('features', {}).get('use_wc_days', False),
            'method': 'Days method (DSO/DIO/DPO)' if proj_yaml.get('features', {}).get('use_wc_days') else '% метод',
            'required': True
        },
        'Debt': {
            'driver': proj_yaml.get('features', {}).get('use_debt_rc', False),
            'method': 'Debt/RC schedule с итерациями' if proj_yaml.get('features', {}).get('use_debt_rc') else 'Упрощенная логика',
            'required': True
        },
        'Tax': {
            'driver': proj_yaml.get('features', {}).get('use_tax_schedule', False),
            'method': 'Tax schedule (DTA/DTL)' if proj_yaml.get('features', {}).get('use_tax_schedule') else 'Effective rate',
            'required': True
        }
    }
    
    for article, check_info in driver_checks.items():
        has_driver = check_info['driver'] is not None and check_info['driver'] != False
        status = '✅' if has_driver or check_info['method'] else '⚠️'
        
        driver_tests.append({
            'Статья': article,
            'Драйвер/Логика': 'Есть' if has_driver else 'Базовая логика',
            'Метод': check_info['method'],
            'Статус': status
        })
    
    driver_df = pd.DataFrame(driver_tests)
    display(driver_df)
    
    # Проверяем наличие макрофакторов для Revenue
    display(Markdown("#### 📊 Проверка макрофакторов для Revenue"))
    
    macro_factors = proj_yaml.get('macro_forecast', {}).get('factors', [])
    if macro_factors:
        macro_check = []
        macro_forecast_dir = croot / 'outputs' / 'macro_forecast'
        
        for factor in macro_factors[:10]:  # Первые 10 факторов
            forecast_file = macro_forecast_dir / f"{factor}_forecast.csv"
            fitted_file = macro_forecast_dir / f"{factor}_ecm_fitted_*.csv"
            has_forecast = forecast_file.exists() or len(list(macro_forecast_dir.glob(f"{factor}_ecm_fitted_*.csv"))) > 0
            
            macro_check.append({
                'Фактор': factor,
                'Прогноз': '✅' if has_forecast else '❌',
                'Файл': forecast_file.name if forecast_file.exists() else 'Не найден'
            })
        
        macro_df = pd.DataFrame(macro_check)
        display(macro_df)
    else:
        print("⚠️ Макрофакторы не настроены в project.yaml")
else:
    print("⚠️ Не удалось загрузить конфигурацию проекта")


In [ ]:
display(Markdown("### ✅ Тест 6: Контрольные точки (Acceptance Checks)"))

# Запускаем acceptance checks если они еще не выполнены
from engine.acceptance.checks import run_acceptance

print("Запуск acceptance checks...")
try:
    run_acceptance(croot)
    print("✅ Acceptance checks выполнены")
except Exception as e:
    print(f"⚠️ Ошибка выполнения acceptance checks: {e}")

# Загружаем результаты acceptance checks
acceptance_df = pd.DataFrame()
mart = _open_data_mart()
if mart is not None:
    try:
        acceptance_df = mart.get_output('acceptance_checks')
        if not acceptance_df.empty:
            print("✅ Acceptance checks загружены из витрины")
    except Exception as e:
        print(f"⚠️ Не удалось загрузить acceptance checks из витрины: {e}")
    finally:
        mart.close()

if acceptance_df.empty:
    acceptance_path = croot / 'outputs' / 'checks' / 'acceptance_checks.csv'
    if acceptance_path.exists():
        acceptance_df = pd.read_csv(acceptance_path)
        print("✅ Acceptance checks загружены из CSV")

if not acceptance_df.empty:
    # Группируем по типам проверок
    if 'check' in acceptance_df.columns and 'ok' in acceptance_df.columns:
        check_summary = acceptance_df.groupby('check').agg({
            'ok': ['sum', 'count']
        }).reset_index()
        check_summary.columns = ['Проверка', 'Успешно', 'Всего']
        check_summary['Процент успеха'] = (check_summary['Успешно'] / check_summary['Всего'] * 100).round(1)
        check_summary['Статус'] = check_summary.apply(
            lambda row: '✅' if row['Процент успеха'] == 100.0 else ('⚠️' if row['Процент успеха'] >= 80.0 else '❌'),
            axis=1
        )
        
        display(check_summary)
        
        # Детальная таблица по годам
        display(Markdown("#### 📊 Детальные результаты по годам"))
        if 'year' in acceptance_df.columns:
            pivot_checks = acceptance_df.pivot_table(
                index='check', 
                columns='year', 
                values='ok', 
                aggfunc='first'
            )
            # Заменяем True/False на ✅/❌
            pivot_checks = pivot_checks.replace({True: '✅', False: '❌'})
            display(pivot_checks)
    else:
        display(acceptance_df.head(20))
else:
    print("⚠️ Acceptance checks не найдены. Запустите build_model() для генерации.")


In [ ]:
display(Markdown("### ✅ Тест 7: Проверка логики моделирования - формулы и связи"))

logic_tests = []

# Убеждаемся что данные загружены
if 'forecast_is' not in locals() or forecast_is.empty:
    forecast_is = load_model_forecast_from_db('IS', canonical=True)

if not forecast_is.empty:
    if 'is_wide' not in locals():
        if 'year' in forecast_is.columns and 'metric' in forecast_is.columns:
            is_wide = forecast_is.pivot(index='metric', columns='year', values='value')
        else:
            is_wide = forecast_is.set_index('metric') if 'metric' in forecast_is.columns else forecast_is
    
    forecast_year_cols = sorted([int(c) for c in is_wide.columns if str(c).isdigit() and int(c) > 2024])
    
    # Загружаем конфигурацию если еще не загружена
    if 'proj_yaml' not in locals():
        proj_yaml_path = croot / 'configs' / 'project.yaml'
        if proj_yaml_path.exists():
            with open(proj_yaml_path, 'r', encoding='utf-8') as f:
                proj_yaml = yaml.safe_load(f)
        else:
            proj_yaml = {}
    
    # 1. EBITDA = Revenue - COGS - SG&A
    ebitda_errors = []
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        revenue = pd.to_numeric(is_wide.loc['revenue', y_str], errors='coerce')
        cogs = pd.to_numeric(is_wide.loc['cogs', y_str], errors='coerce')
        sga = pd.to_numeric(is_wide.loc['sga', y_str], errors='coerce')
        ebitda = pd.to_numeric(is_wide.loc['ebitda', y_str], errors='coerce')
        
        if all(pd.notna(v) for v in [revenue, cogs, sga, ebitda]):
            calculated_ebitda = revenue - cogs - sga
            diff = abs(ebitda - calculated_ebitda)
            if diff > 1.0:
                ebitda_errors.append(f"{y}: diff=${diff:,.0f}")
    
    logic_tests.append({
        'Формула': 'EBITDA = Revenue - COGS - SG&A',
        'Статус': '✅' if not ebitda_errors else '❌',
        'Ошибки': ', '.join(ebitda_errors) if ebitda_errors else 'Нет'
    })

# 2. EBIT = EBITDA - Depreciation - Amortization
if not forecast_is.empty:
    ebit_errors = []
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        ebitda = pd.to_numeric(is_wide.loc['ebitda', y_str], errors='coerce')
        dep = pd.to_numeric(is_wide.loc['depreciation_owned', y_str], errors='coerce') or 0.0
        amort = pd.to_numeric(is_wide.loc['amortization', y_str], errors='coerce') or 0.0
        ebit = pd.to_numeric(is_wide.loc['ebit', y_str], errors='coerce')
        
        if all(pd.notna(v) for v in [ebitda, ebit]):
            calculated_ebit = ebitda - dep - amort
            diff = abs(ebit - calculated_ebit)
            if diff > 1.0:
                ebit_errors.append(f"{y}: diff=${diff:,.0f}")
    
    logic_tests.append({
        'Формула': 'EBIT = EBITDA - Depreciation - Amortization',
        'Статус': '✅' if not ebit_errors else '❌',
        'Ошибки': ', '.join(ebit_errors) if ebit_errors else 'Нет'
    })

# 3. EBT = EBIT + Interest Income - Interest Expense - Lease Interest
if not forecast_is.empty:
    ebt_errors = []
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        ebit = pd.to_numeric(is_wide.loc['ebit', y_str], errors='coerce')
        int_income = pd.to_numeric(is_wide.loc['interest_income', y_str], errors='coerce') or 0.0
        int_expense = pd.to_numeric(is_wide.loc['interest_expense', y_str], errors='coerce') or 0.0
        lease_int = pd.to_numeric(is_wide.loc['lease_interest', y_str], errors='coerce') or 0.0
        ebt = pd.to_numeric(is_wide.loc['ebt', y_str], errors='coerce')
        
        if pd.notna(ebit) and pd.notna(ebt):
            calculated_ebt = ebit + int_income - int_expense - lease_int
            diff = abs(ebt - calculated_ebt)
            if diff > 1.0:
                ebt_errors.append(f"{y}: diff=${diff:,.0f}")
    
    logic_tests.append({
        'Формула': 'EBT = EBIT + Interest Income - Interest Expense - Lease Interest',
        'Статус': '✅' if not ebt_errors else '❌',
        'Ошибки': ', '.join(ebt_errors) if ebt_errors else 'Нет'
    })

# 4. Net Income = EBT - Tax Expense
if not forecast_is.empty:
    ni_errors = []
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        ebt = pd.to_numeric(is_wide.loc['ebt', y_str], errors='coerce')
        tax = pd.to_numeric(is_wide.loc['tax_expense', y_str], errors='coerce') or 0.0
        ni = pd.to_numeric(is_wide.loc['net_income', y_str], errors='coerce')
        
        if pd.notna(ebt) and pd.notna(ni):
            calculated_ni = ebt - tax
            diff = abs(ni - calculated_ni)
            if diff > 1.0:
                ni_errors.append(f"{y}: diff=${diff:,.0f}")
    
    logic_tests.append({
        'Формула': 'Net Income = EBT - Tax Expense',
        'Статус': '✅' if not ni_errors else '❌',
        'Ошибки': ', '.join(ni_errors) if ni_errors else 'Нет'
    })

# 5. COGS Ratio в допустимых пределах (0.55 - 0.98)
if not forecast_is.empty:
    cogs_clamp_errors = []
    cogs_config = proj_yaml.get('model', {}).get('standard', {}).get('cogs', {})
    cogs_clamp_min = cogs_config.get('clamp', {}).get('min', 0.55)
    cogs_clamp_max = cogs_config.get('clamp', {}).get('max', 0.98)
    
    for y in forecast_year_cols[:5]:
        y_str = str(y)
        revenue = pd.to_numeric(is_wide.loc['revenue', y_str], errors='coerce')
        cogs = pd.to_numeric(is_wide.loc['cogs', y_str], errors='coerce')
        
        if pd.notna(revenue) and pd.notna(cogs) and revenue > 0:
            cogs_ratio = cogs / revenue
            if not (cogs_clamp_min <= cogs_ratio <= cogs_clamp_max):
                cogs_clamp_errors.append(f"{y}: ratio={cogs_ratio:.2%} (допустимо {cogs_clamp_min:.0%}-{cogs_clamp_max:.0%})")
    
    logic_tests.append({
        'Формула': f'COGS Ratio в пределах {cogs_clamp_min:.0%}-{cogs_clamp_max:.0%}',
        'Статус': '✅' if not cogs_clamp_errors else '❌',
        'Ошибки': ', '.join(cogs_clamp_errors) if cogs_clamp_errors else 'Нет'
    })

if logic_tests:
    logic_df = pd.DataFrame(logic_tests)
    display(logic_df)
else:
    print("⚠️ Не удалось проверить логику моделирования - данные отсутствуют")


In [ ]:
display(Markdown("### 📊 Итоговая сводка всех тестов"))

# Собираем результаты всех тестов
final_summary = {
    'Категория теста': [],
    'Пройдено': [],
    'Всего': [],
    'Статус': []
}

# 1. Полнота данных
if 'completeness_df' in locals() and not completeness_df.empty:
    passed = len(completeness_df[completeness_df['Статус'] == '✅'])
    total = len(completeness_df)
    final_summary['Категория теста'].append('Полнота данных (IS/BS/CF)')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

# 2. Корректность Corkscrews
if 'corkscrew_df' in locals() and not corkscrew_df.empty:
    passed = len(corkscrew_df[corkscrew_df['Статус'] == '✅'])
    total = len(corkscrew_df)
    final_summary['Категория теста'].append('Корректность Corkscrews')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

# 3. Финансовые связи
if 'financial_df' in locals() and not financial_df.empty:
    passed = len(financial_df[financial_df['Статус'] == '✅'])
    total = len(financial_df)
    final_summary['Категория теста'].append('Финансовые связи')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

# 4. Соответствие трендам
if 'trend_df' in locals() and not trend_df.empty:
    passed = len(trend_df[trend_df['Статус'] == '✅'])
    total = len(trend_df)
    final_summary['Категория теста'].append('Соответствие историческим трендам')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

# 5. Наличие драйверов
if 'driver_df' in locals() and not driver_df.empty:
    passed = len(driver_df[driver_df['Статус'] == '✅'])
    total = len(driver_df)
    final_summary['Категория теста'].append('Наличие драйверов и логики')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

# 6. Acceptance Checks
if 'check_summary' in locals() and not check_summary.empty:
    passed = len(check_summary[check_summary['Статус'] == '✅'])
    total = len(check_summary)
    final_summary['Категория теста'].append('Acceptance Checks')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

# 7. Логика моделирования
if 'logic_df' in locals() and not logic_df.empty:
    passed = len(logic_df[logic_df['Статус'] == '✅'])
    total = len(logic_df)
    final_summary['Категория теста'].append('Логика моделирования (формулы)')
    final_summary['Пройдено'].append(passed)
    final_summary['Всего'].append(total)
    final_summary['Статус'].append('✅' if passed == total else ('⚠️' if passed > 0 else '❌'))

if final_summary['Категория теста']:
    summary_df = pd.DataFrame(final_summary)
    display(summary_df)
    
    # Общая статистика
    total_passed = sum(final_summary['Пройдено'])
    total_tests = sum(final_summary['Всего'])
    success_rate = (total_passed / total_tests * 100) if total_tests > 0 else 0
    
    print(f"\n📊 Итоговая статистика тестирования:")
    print(f"  Всего тестов: {total_tests}")
    print(f"  Пройдено: {total_passed}")
    print(f"  Процент успеха: {success_rate:.1f}%")
    print(f"  Статус: {'✅ Модель готова' if success_rate >= 90 else ('⚠️ Требуется доработка' if success_rate >= 70 else '❌ Критические ошибки')}")
else:
    print("⚠️ Тесты не выполнены. Запустите build_model() для генерации данных.")


## 4️⃣ Проверка выходных файлов <a id="section-4"></a>

In [ ]:
display(Markdown("### 📊 Сводная таблица 3-Statement (последние 5 лет истории + прогноз)"))

# Маппинг метрик для поиска в прогнозе (короткие названия -> канонические)
METRIC_MAPPING_IS = {
    'depreciation_owned': ['dep_pp&e', 'dep_ppe', 'depreciation_owned'],
    'depreciation_rou': ['dep_rou', 'depreciation_rou'],
    'total_da': ['dep_total', 'total_da', 'depreciation_and_amortization'],
    'interest_expense': ['interest_expense_debt', 'interest_expense'],
    'tax': ['tax_expense', 'current_tax', 'tax'],
    'amortization': ['amortization']  # Может отсутствовать в прогнозе
}

METRIC_MAPPING_BS = {
    'ar': ['accounts_receivable', 'ar'],
    'ap': ['accounts_payable', 'ap'],
    'ppe': ['ppe_net', 'ppe'],
    'equity': ['total_equity', 'equity'],
    'debt': ['long_term_debt', 'lt_debt', 'debt'],  # Для debt нужно суммировать ST + LT
    'lease_liability': ['current_lease_liability', 'long_term_lease_liability', 'lease_liability'],
    'st_debt': ['short_term_debt', 'st_debt'],
    'dta': ['deferred_tax_assets', 'dta'],
    'dtl': ['deferred_tax_liabilities', 'dtl']
}

METRIC_MAPPING_CF = {
    'depreciation': ['depreciation', 'depreciation_owned', 'dep_pp&e'],
    'amortization': ['amortization'],
    'ar_delta': ['wc_accounts_receivable_change', 'ar_delta'],
    'inventory_delta': ['wc_inventory_change', 'inventory_delta'],
    'ap_delta': ['wc_accounts_payable_change', 'ap_delta'],
    'total_capex': ['capex', 'total_capex'],
    'maint_capex': ['maint_capex'],
    'growth_capex': ['growth_capex']
}

def find_metric_in_forecast(forecast_df, metric_name, statement_type='IS'):
    """Находит метрику в прогнозе с учетом маппинга"""
    if forecast_df.empty or 'metric' not in forecast_df.columns:
        return pd.DataFrame()
    
    # Прямой поиск
    result = forecast_df[forecast_df['metric'].str.lower() == metric_name.lower()]
    if not result.empty:
        return result
    
    # Поиск через маппинг
    mapping = {}
    if statement_type == 'IS':
        mapping = METRIC_MAPPING_IS
    elif statement_type == 'BS':
        mapping = METRIC_MAPPING_BS
    elif statement_type == 'CF':
        mapping = METRIC_MAPPING_CF
    
    if metric_name in mapping:
        for alias in mapping[metric_name]:
            result = forecast_df[forecast_df['metric'].str.lower() == alias.lower()]
            if not result.empty:
                return result
    
    # Частичное совпадение
    result = forecast_df[forecast_df['metric'].str.lower().str.contains(metric_name.lower(), na=False, regex=False)]
    if not result.empty:
        return result
    
    return pd.DataFrame()

# Загружаем историю и прогноз (БД имеет приоритет над CSV)
hist_is = load_history_from_db('IS')
hist_bs = load_history_from_db('BS')
hist_cf = load_history_from_db('CF')
is_forecast = load_model_forecast_from_db('IS', canonical=True)
bs_forecast = load_model_forecast_from_db('BS', canonical=True)
cf_forecast = load_model_forecast_from_db('CF', canonical=True)

# Последние 5 лет истории (поддержка wide format с метрикой как индекс)
if 'year' in hist_is.columns:
    history_years = sorted([y for y in hist_is['year'].values if pd.notna(y)])[-5:]
else:
    # Wide format: годы это колонки
    year_cols = [c for c in hist_is.columns if str(c).isdigit() and len(str(c)) == 4]
    history_years = sorted([int(c) for c in year_cols])[-5:]
forecast_years = sorted([int(c) for c in is_forecast.columns if str(c).isdigit() and len(str(c)) == 4])
all_years_display = list(history_years) + forecast_years

print(f"\n📅 Отчетные периоды: {history_years} (история) + {forecast_years} (прогноз)")

# === INCOME STATEMENT ===
display(Markdown("#### 📈 Income Statement (Отчет о прибылях и убытках)"))

# Собираем IS данные
is_data = []
metrics_is = ['revenue', 'cogs', 'sga', 'depreciation_owned', 'depreciation_rou', 'amortization', 'total_da', 'ebitda', 'ebit', 'gain_loss_on_disposal', 'interest_income', 'interest_expense', 'lease_interest', 'tax', 'net_income']

for metric in metrics_is:
    row_data = {'Metric': metric.upper().replace('_', ' ')}
    
    # История - рассчитываем отсутствующие метрики
    for y in history_years:
        hist_row = hist_is[hist_is['year'] == y]
        if not hist_row.empty:
            if metric == 'revenue':
                val = hist_row['revenue'].iloc[0] if 'revenue' in hist_is.columns else None
            elif metric == 'cogs':
                val = hist_row['cogs'].iloc[0] if 'cogs' in hist_is.columns else None
            elif metric == 'sga':
                val = hist_row['sga'].iloc[0] if 'sga' in hist_is.columns else None
            elif metric == 'dep':
                val = hist_row['depreciation'].iloc[0] if 'depreciation' in hist_is.columns else None
            elif metric == 'ebitda':
                # EBITDA = Revenue - COGS - SG&A
                rev = hist_row['revenue'].iloc[0] if 'revenue' in hist_is.columns else None
                cgs = hist_row['cogs'].iloc[0] if 'cogs' in hist_is.columns else None
                sg = hist_row['sga'].iloc[0] if 'sga' in hist_is.columns else None
                val = rev - cgs - sg if all(pd.notna(v) for v in [rev, cgs, sg]) else None
            elif metric == 'ebit':
                # EBIT = EBITDA - Depreciation
                rev = hist_row['revenue'].iloc[0] if 'revenue' in hist_is.columns else None
                cgs = hist_row['cogs'].iloc[0] if 'cogs' in hist_is.columns else None
                sg = hist_row['sga'].iloc[0] if 'sga' in hist_is.columns else None
                dep = hist_row['depreciation'].iloc[0] if 'depreciation' in hist_is.columns else None
                val = rev - cgs - sg - dep if all(pd.notna(v) for v in [rev, cgs, sg, dep]) else None
            elif metric == 'interest_income':
                val = hist_row.get('interest_income', hist_row.get('finance_income', None)).iloc[0] if 'interest_income' in hist_is.columns or 'finance_income' in hist_is.columns else None
            elif metric == 'interest_expense':
                val = hist_row.get('interest_expense', hist_row.get('finance_expense', None)).iloc[0] if 'interest_expense' in hist_is.columns or 'finance_expense' in hist_is.columns else None
            elif metric == 'lease_interest':
                val = 0.0  # В истории лизинг не учитывается
            elif metric == 'depreciation_owned':
                # Depreciation Owned (с fallback на dep или depreciation)
                if 'depreciation_owned' in hist_is.columns:
                    val = hist_row['depreciation_owned'].iloc[0]
                elif 'dep' in hist_is.columns:
                    dep_total = hist_row['dep'].iloc[0]
                    dep_rou = hist_row.get('depreciation_rou', pd.Series([0.0])).iloc[0] if 'depreciation_rou' in hist_is.columns else 0.0
                    val = dep_total - dep_rou
                else:
                    val = None
            elif metric == 'depreciation_rou':
                val = hist_row['depreciation_rou'].iloc[0] if 'depreciation_rou' in hist_is.columns else 0.0
            elif metric == 'amortization':
                val = hist_row['amortization'].iloc[0] if 'amortization' in hist_is.columns else None
            elif metric == 'total_da':
                # Total D&A = Dep Owned + Dep ROU + Amortization
                dep_owned_val = hist_row.get('depreciation_owned', pd.Series([0.0])).iloc[0] if 'depreciation_owned' in hist_is.columns else 0.0
                dep_rou_val = hist_row.get('depreciation_rou', pd.Series([0.0])).iloc[0] if 'depreciation_rou' in hist_is.columns else 0.0
                amort_val = hist_row.get('amortization', pd.Series([0.0])).iloc[0] if 'amortization' in hist_is.columns else 0.0
                val = dep_owned_val + dep_rou_val + amort_val
            elif metric == 'gain_loss_on_disposal':
                val = hist_row['gain_loss_on_disposal'].iloc[0] if 'gain_loss_on_disposal' in hist_is.columns else 0.0
            elif metric == 'tax':
                # Используем income_tax_expense из истории
                if 'income_tax_expense' in hist_is.columns:
                    val = hist_row['income_tax_expense'].iloc[0]
                elif 'tax' in hist_is.columns:
                    val = hist_row['tax'].iloc[0]
                else:
                    val = None
            elif metric == 'net_income':
                val = hist_row['net_income'].iloc[0] if 'net_income' in hist_is.columns else None
            else:
                val = None
        else:
            val = None
        row_data[y] = f"${val:,.0f}" if pd.notna(val) else "-"
    
    # Прогноз - используем функцию поиска с маппингом
    for y in forecast_years:
        forecast_row = find_metric_in_forecast(is_forecast, metric, statement_type='IS')
        if not forecast_row.empty:
            # Пробуем str и int колонку
            val = None
            if str(y) in forecast_row.columns:
                val = forecast_row[str(y)].iloc[0]
            elif y in forecast_row.columns:
                val = forecast_row[y].iloc[0]
            # Для total_da, если не найдено, вычисляем из компонентов
            if metric == 'total_da' and (pd.isna(val) or val == 0):
                dep_owned_row = find_metric_in_forecast(is_forecast, 'depreciation_owned', 'IS')
                dep_rou_row = find_metric_in_forecast(is_forecast, 'depreciation_rou', 'IS')
                amort_row = find_metric_in_forecast(is_forecast, 'amortization', 'IS')
                dep_owned_val = dep_owned_row[str(y)].iloc[0] if not dep_owned_row.empty and str(y) in dep_owned_row.columns else 0.0
                dep_rou_val = dep_rou_row[str(y)].iloc[0] if not dep_rou_row.empty and str(y) in dep_rou_row.columns else 0.0
                amort_val = amort_row[str(y)].iloc[0] if not amort_row.empty and str(y) in amort_row.columns else 0.0
                val = dep_owned_val + dep_rou_val + amort_val
            
            # Для EBIT вычисляем: EBITDA - Depreciation
            if metric == 'ebit' and (pd.isna(val) or val == 0):
                ebitda_row = find_metric_in_forecast(is_forecast, 'ebitda', 'IS')
                dep_row = find_metric_in_forecast(is_forecast, 'depreciation_owned', 'IS')
                ebitda_val = ebitda_row[str(y)].iloc[0] if not ebitda_row.empty and str(y) in ebitda_row.columns else None
                dep_val = dep_row[str(y)].iloc[0] if not dep_row.empty and str(y) in dep_row.columns else 0.0
                if pd.notna(ebitda_val):
                    val = ebitda_val - (dep_val if pd.notna(dep_val) else 0.0)
            
            # Для tax используем tax_expense
            if metric == 'tax' and (pd.isna(val) or val == 0):
                tax_row = find_metric_in_forecast(is_forecast, 'tax_expense', 'IS')
                if not tax_row.empty and str(y) in tax_row.columns:
                    val = tax_row[str(y)].iloc[0]
            
            # Для некоторых метрик нулевые значения валидны (tax может быть 0 при убытке)
            show_zero = metric in ['tax']
            if pd.notna(val):
                if val != 0 or show_zero:
                    row_data[y] = f"${val:,.0f}"
                else:
                    row_data[y] = "-"
            else:
                row_data[y] = "-"
        else:
            row_data[y] = "-"
    is_data.append(row_data)

is_df = pd.DataFrame(is_data)
display(is_df)

# === BALANCE SHEET ===
display(Markdown("\n#### 🏦 Balance Sheet (Баланс)"))
# Полная каноническая форма Balance Sheet
metrics_bs = [
    'cash', 
    'ar',
    'inventory',
    'other_current_assets',
    'ppe',
    'intangibles',
    'rou_asset',
    'dta',
    'other_non_current_assets',
    'total_assets',
    'st_debt',
    'ap',
    'accrued_liabilities',
    'other_current_liabilities',
    'debt',
    'lease_liability',
    'dtl',
    'provisions',
    'other_non_current_liabilities',
    'equity',
    'total_liab_equity'
]
bs_data = []
for metric in metrics_bs:
    row_data = {'Metric': metric.upper().replace('_', ' ')}

    # История - маппинг имен колонок
    metric_hist_mapping = {
        'ppe': 'ppe_net',
        'ar': 'accounts_receivable',
        'ap': 'accounts_payable',
        'equity': 'total_equity',
        'debt': 'lt_debt'  # debt = lt_debt + st_debt (только LT в истории, т.к. ST отдельно)
    }
    metric_hist = metric_hist_mapping.get(metric, metric)
    
    for y in history_years:
        hist_row = hist_bs[hist_bs['year'] == y]
        if metric_hist in hist_bs.columns:
            val = hist_row[metric_hist].iloc[0] if not hist_row.empty else None
            # Для debt добавляем st_debt если это debt metric
            if metric == 'debt' and 'st_debt' in hist_bs.columns:
                st_val = hist_row['st_debt'].iloc[0] if not hist_row.empty else 0.0
                val = (val or 0.0) + (st_val if pd.notna(st_val) else 0.0)
            row_data[y] = f"${val:,.0f}" if pd.notna(val) else "-"
        else:
            row_data[y] = "-"
    
    # Прогноз - используем функцию поиска с маппингом
    for y in forecast_years:
        forecast_row = find_metric_in_forecast(bs_forecast, metric, statement_type='BS')
        if not forecast_row.empty:
            # Пробуем str и int колонку
            val = None
            if str(y) in forecast_row.columns:
                val = forecast_row[str(y)].iloc[0]
            elif y in forecast_row.columns:
                val = forecast_row[y].iloc[0]
            
            # Для debt суммируем ST + LT
            if metric == 'debt':
                st_row = find_metric_in_forecast(bs_forecast, 'st_debt', 'BS')
                lt_row = find_metric_in_forecast(bs_forecast, 'long_term_debt', 'BS')
                st_val = st_row[str(y)].iloc[0] if not st_row.empty and str(y) in st_row.columns else 0.0
                lt_val = lt_row[str(y)].iloc[0] if not lt_row.empty and str(y) in lt_row.columns else 0.0
                val = (st_val if pd.notna(st_val) else 0.0) + (lt_val if pd.notna(lt_val) else 0.0)
            
            # Для lease_liability суммируем current + long_term
            if metric == 'lease_liability':
                cur_row = find_metric_in_forecast(bs_forecast, 'current_lease_liability', 'BS')
                lt_row = find_metric_in_forecast(bs_forecast, 'long_term_lease_liability', 'BS')
                cur_val = cur_row[str(y)].iloc[0] if not cur_row.empty and str(y) in cur_row.columns else 0.0
                lt_val = lt_row[str(y)].iloc[0] if not lt_row.empty and str(y) in lt_row.columns else 0.0
                val = (cur_val if pd.notna(cur_val) else 0.0) + (lt_val if pd.notna(lt_val) else 0.0)
            
            # Для total_assets вычисляем сумму всех активов
            if metric == 'total_assets':
                cash_row = find_metric_in_forecast(bs_forecast, 'cash', 'BS')
                ar_row = find_metric_in_forecast(bs_forecast, 'accounts_receivable', 'BS')
                inv_row = find_metric_in_forecast(bs_forecast, 'inventory', 'BS')
                oca_row = find_metric_in_forecast(bs_forecast, 'other_current_assets', 'BS')
                ppe_row = find_metric_in_forecast(bs_forecast, 'ppe_net', 'BS')
                int_row = find_metric_in_forecast(bs_forecast, 'intangibles', 'BS')
                rou_row = find_metric_in_forecast(bs_forecast, 'rou_asset', 'BS')
                dta_row = find_metric_in_forecast(bs_forecast, 'deferred_tax_assets', 'BS')
                onca_row = find_metric_in_forecast(bs_forecast, 'other_non_current_assets', 'BS')
                
                val = 0.0
                for row in [cash_row, ar_row, inv_row, oca_row, ppe_row, int_row, rou_row, dta_row, onca_row]:
                    if not row.empty and str(y) in row.columns:
                        v = row[str(y)].iloc[0]
                        if pd.notna(v):
                            val += float(v)
            
            # Для total_equity вычисляем сумму компонентов equity
            if metric == 'equity' or metric == 'total_equity':
                sc_row = find_metric_in_forecast(bs_forecast, 'share_capital', 'BS')
                apic_row = find_metric_in_forecast(bs_forecast, 'apic', 'BS')
                re_row = find_metric_in_forecast(bs_forecast, 'retained_earnings', 'BS')
                aoci_row = find_metric_in_forecast(bs_forecast, 'aoci', 'BS')
                nci_row = find_metric_in_forecast(bs_forecast, 'nci', 'BS')
                
                val = 0.0
                for row in [sc_row, apic_row, re_row, aoci_row, nci_row]:
                    if not row.empty and str(y) in row.columns:
                        v = row[str(y)].iloc[0]
                        if pd.notna(v):
                            val += float(v)
            
            # Для total_liab_equity суммируем liabilities + equity
            if metric == 'total_liab_equity':
                liab_row = find_metric_in_forecast(bs_forecast, 'total_liabilities', 'BS')
                equity_row = find_metric_in_forecast(bs_forecast, 'total_equity', 'BS')
                liab_val = liab_row[str(y)].iloc[0] if not liab_row.empty and str(y) in liab_row.columns else 0.0
                equity_val = equity_row[str(y)].iloc[0] if not equity_row.empty and str(y) in equity_row.columns else 0.0
                val = (liab_val if pd.notna(liab_val) else 0.0) + (equity_val if pd.notna(equity_val) else 0.0)
            
            row_data[y] = f"${val:,.0f}" if pd.notna(val) and val != 0 else "-"
        else:
            row_data[y] = "-"
    bs_data.append(row_data)

bs_df = pd.DataFrame(bs_data)
display(bs_df)

# === CASH FLOW ===
display(Markdown("\n#### 💰 Cash Flow Statement (Отчет о движении денежных средств)"))
# Полный канонический формат CF_detail
metrics_cf = [
    'net_income', 'depreciation', 'amortization', 'interest_income', 'interest_expense', 'tax_paid',
    'wc_delta', 'ar_delta', 'inventory_delta', 'ap_delta', 
    'lease_payments_cfo', 'lease_interest_cfo', 'cfo',
    'maint_capex', 'growth_capex', 'total_capex', 'disposals', 'ppe_balance', 'intangibles_balance', 'cfi',
    'debt_borrowings', 'debt_repayments', 'rc_draws', 'rc_repayments', 
    'lease_payments_cff', 'dividends', 'cff', 'net_change'
]
cf_data = []
for metric in metrics_cf:
    row_data = {'Metric': metric.upper().replace('_', ' ')}
    
    # История
    cf_col = metric.lower()

    for y in history_years:
        hist_row = hist_cf[hist_cf['year'] == y]
        if cf_col in hist_cf.columns:
            val = hist_row[cf_col].iloc[0] if not hist_row.empty else None
            row_data[y] = f"${val:,.0f}" if pd.notna(val) else "-"
        else:
            row_data[y] = "-"
    
    # Прогноз - используем функцию поиска с маппингом
    for y in forecast_years:
        forecast_row = find_metric_in_forecast(cf_forecast, metric, statement_type='CF')
        if not forecast_row.empty:
            # Пробуем str и int колонку
            val = None
            if str(y) in forecast_row.columns:
                val = forecast_row[str(y)].iloc[0]
            elif y in forecast_row.columns:
                val = forecast_row[y].iloc[0]
            
            # Для total_capex, если не найдено, используем capex
            if metric == 'total_capex' and (pd.isna(val) or val == 0):
                capex_row = find_metric_in_forecast(cf_forecast, 'capex', 'CF')
                if not capex_row.empty and str(y) in capex_row.columns:
                    val = capex_row[str(y)].iloc[0]
            
            row_data[y] = f"${val:,.0f}" if pd.notna(val) else "-"
        else:
            row_data[y] = "-"
    cf_data.append(row_data)

cf_df = pd.DataFrame(cf_data)
display(cf_df)

print(f"\n✅ Сводная таблица 3-Statement построена")
print(f"📊 Периоды: {min(all_years_display)}-{max(all_years_display)} ({len(all_years_display)} лет)")


### 🔍 Детальный анализ сопоставимости истории и прогноза

Анализ каждой статьи с проверкой:
- Знаков (положительные/отрицательные)
- Порядка чисел (масштаб)
- Логики изменений
- Сопоставимости величин

In [ ]:
# Детальный анализ сопоставимости истории и прогноза
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

display(Markdown("### 🔍 Детальный анализ каждой статьи: История vs Прогноз"))

# Загружаем данные если еще не загружены
if 'hist_is' not in locals() or hist_is.empty:
    hist_is = load_history_from_db('IS', canonical=True)
if 'hist_bs' not in locals() or hist_bs.empty:
    hist_bs = load_history_from_db('BS', canonical=True)
if 'hist_cf' not in locals() or hist_cf.empty:
    hist_cf = load_history_from_db('CF', canonical=True)

if 'forecast_is' not in locals() or forecast_is.empty:
    forecast_is = load_model_forecast_from_db('IS', canonical=True)
if 'forecast_bs' not in locals() or forecast_bs.empty:
    forecast_bs = load_model_forecast_from_db('BS', canonical=True)
if 'forecast_cf' not in locals() or forecast_cf.empty:
    forecast_cf = load_model_forecast_from_db('CF', canonical=True)

# Преобразуем в wide format
def to_wide(df, metric_col='metric'):
    if df.empty:
        return pd.DataFrame()
    if 'year' in df.columns and metric_col in df.columns:
        return df.pivot(index=metric_col, columns='year', values='value')
    elif metric_col in df.columns:
        return df.set_index(metric_col)
    return df

hist_is_wide = to_wide(hist_is)
hist_bs_wide = to_wide(hist_bs)
hist_cf_wide = to_wide(hist_cf)
forecast_is_wide = to_wide(forecast_is)
forecast_bs_wide = to_wide(forecast_bs)
forecast_cf_wide = to_wide(forecast_cf)

# Получаем годы
hist_years = sorted([int(c) for c in hist_is_wide.columns if str(c).isdigit() and int(c) <= 2024])[-5:]
forecast_years = sorted([int(c) for c in forecast_is_wide.columns if str(c).isdigit() and int(c) > 2024])

def get_value(df, metric, year):
    """Безопасное получение значения"""
    if df.empty or metric not in df.index:
        return None
    year_str = str(year)
    if year_str not in df.columns:
        return None
    val = pd.to_numeric(df.loc[metric, year_str], errors='coerce')
    return float(val) if pd.notna(val) else None

def analyze_item(item_name, hist_df, forecast_df):
    """Анализирует одну статью"""
    issues = []
    warnings = []
    
    in_hist = item_name in hist_df.index if not hist_df.empty else False
    in_forecast = item_name in forecast_df.index if not forecast_df.empty else False
    
    if not in_hist and not in_forecast:
        return None
    
    # Получаем последние значения истории и первые прогноза
    hist_vals = {}
    forecast_vals = {}
    
    if in_hist:
        for y in hist_years:
            val = get_value(hist_df, item_name, y)
            if val is not None:
                hist_vals[y] = val
    
    if in_forecast:
        for y in forecast_years:
            val = get_value(forecast_df, item_name, y)
            if val is not None:
                forecast_vals[y] = val
    
    if not hist_vals and not forecast_vals:
        return None
    
    result = {
        'Статья': item_name,
        'В истории': '✅' if in_hist else '❌',
        'В прогнозе': '✅' if in_forecast else '❌',
        'Проблемы': [],
        'Предупреждения': []
    }
    
    if hist_vals and forecast_vals:
        # 1. Проверка знаков
        hist_nonzero = [v for v in hist_vals.values() if abs(v) > 1e-6]
        forecast_nonzero = [v for v in forecast_vals.values() if abs(v) > 1e-6]
        
        if hist_nonzero and forecast_nonzero:
            hist_signs = [1 if v > 0 else -1 for v in hist_nonzero]
            forecast_signs = [1 if v > 0 else -1 for v in forecast_nonzero]
            
            hist_typical = max(set(hist_signs), key=hist_signs.count) if hist_signs else 0
            forecast_typical = max(set(forecast_signs), key=forecast_signs.count) if forecast_signs else 0
            
            if hist_typical != 0 and forecast_typical != 0 and hist_typical != forecast_typical:
                result['Проблемы'].append(f"❌ Смена знака: история {('+' if hist_typical > 0 else '-')}, прогноз {('+' if forecast_typical > 0 else '-')}")
        
        # 2. Проверка масштаба
        hist_abs = [abs(v) for v in hist_vals.values() if abs(v) > 1e-6]
        forecast_abs = [abs(v) for v in forecast_vals.values() if abs(v) > 1e-6]
        
        if hist_abs and forecast_abs:
            hist_avg = np.mean(hist_abs)
            forecast_avg = np.mean(forecast_abs)
            
            if hist_avg > 0:
                ratio = forecast_avg / hist_avg
                if ratio > 10 or ratio < 0.1:
                    result['Проблемы'].append(f"❌ Разница в масштабе: {ratio:.1f}x (история ~${hist_avg:,.0f}, прогноз ~${forecast_avg:,.0f})")
                elif ratio > 3 or ratio < 0.33:
                    result['Предупреждения'].append(f"⚠️ Значительное изменение масштаба: {ratio:.1f}x")
        
        # 3. Проверка перехода
        if len(hist_vals) >= 1 and len(forecast_vals) >= 1:
            hist_last = list(hist_vals.values())[-1]
            forecast_first = list(forecast_vals.values())[0]
            
            jump = forecast_first - hist_last
            if abs(hist_last) > 1e-6:
                jump_pct = (jump / abs(hist_last)) * 100
                if abs(jump_pct) > 500:
                    result['Проблемы'].append(f"❌ Резкий скачок на границе: {jump:,.0f} ({jump_pct:+.0f}%)")
                elif abs(jump_pct) > 200:
                    result['Предупреждения'].append(f"⚠️ Большое изменение на границе: {jump:,.0f} ({jump_pct:+.0f}%)")
    
    elif not in_hist and in_forecast:
        result['Предупреждения'].append("⚠️ Статья отсутствует в истории, но есть в прогнозе")
    elif in_hist and not in_forecast:
        result['Предупреждения'].append("⚠️ Статья есть в истории, но отсутствует в прогнозе")
    
    return result

# Анализируем ключевые статьи
is_items = ['revenue', 'cogs', 'sga', 'depreciation_owned', 'depreciation_rou', 'amortization', 
            'ebitda', 'ebit', 'interest_income', 'interest_expense', 'lease_interest', 'tax', 'net_income']
bs_items = ['cash', 'ar', 'inventory', 'ppe', 'intangibles', 'rou_asset', 'total_assets',
            'st_debt', 'ap', 'debt', 'lease_liability', 'equity', 'total_liabilities', 'total_equity']
cf_items = ['net_income', 'depreciation', 'amortization', 'wc_delta', 'cfo', 'capex', 'cfi', 
            'debt_borrowings', 'debt_repayments', 'dividends', 'cff', 'net_change']

is_analysis = [analyze_item(item, hist_is_wide, forecast_is_wide) for item in is_items]
is_analysis = [a for a in is_analysis if a is not None]

bs_analysis = [analyze_item(item, hist_bs_wide, forecast_bs_wide) for item in bs_items]
bs_analysis = [a for a in bs_analysis if a is not None]

cf_analysis = [analyze_item(item, hist_cf_wide, forecast_cf_wide) for item in cf_items]
cf_analysis = [a for a in cf_analysis if a is not None]

# Выводим результаты
def print_analysis(analysis_list, title):
    display(Markdown(f"### {title}"))
    
    for item in analysis_list:
        if item['Проблемы'] or item['Предупреждения']:
            display(Markdown(f"#### {item['Статья']}"))
            print(f"  В истории: {item['В истории']} | В прогнозе: {item['В прогнозе']}")
            
            if item['Проблемы']:
                print(f"  \n  🔴 ПРОБЛЕМЫ:")
                for prob in item['Проблемы']:
                    print(f"    {prob}")
            
            if item['Предупреждения']:
                print(f"  \n  ⚠️ ПРЕДУПРЕЖДЕНИЯ:")
                for warn in item['Предупреждения']:
                    print(f"    {warn}")
            print()

print_analysis(is_analysis, '📊 Income Statement (IS)')
print_analysis(bs_analysis, '🏦 Balance Sheet (BS)')
print_analysis(cf_analysis, '💰 Cash Flow Statement (CF)')

# Сводная таблица
all_issues = []
for analysis in [is_analysis, bs_analysis, cf_analysis]:
    for item in analysis:
        if item['Проблемы'] or item['Предупреждения']:
            all_issues.append({
                'Статья': item['Статья'],
                'Проблем': len(item['Проблемы']),
                'Предупреждений': len(item['Предупреждения']),
                'Статус': '❌' if item['Проблемы'] else '⚠️'
            })

if all_issues:
    display(Markdown("### 📋 Сводная таблица проблем"))
    issues_df = pd.DataFrame(all_issues)
    display(issues_df)
    print(f"\nВсего статей с проблемами: {len(all_issues)}")
    print(f"Критических проблем: {sum(1 for i in all_issues if i['Проблем'] > 0)}")
    print(f"Предупреждений: {sum(1 for i in all_issues if i['Предупреждений'] > 0)}")
else:
    print("✅ Все статьи проверены, критических проблем не обнаружено")
